# Reproducing Knierim, McNaughton & Poe (2000) from dandiset 001754

Paper. Knierim JJ, McNaughton BL, Poe GR (2000). Three-dimensional spatial selectivity of
hippocampal neurons during space flight. *Nature Neuroscience* 3:209-210.
[doi:10.1038/72910](https://doi.org/10.1038/72910). The paper is not open access. This attempt
used only its abstract and the titles of its three figures, from the publisher's page. Four
claims come from the dandiset's own description, which summarises the paper's findings; they
could not be checked against the paper's text.

Dandiset. [DANDI:001754](https://dandiarchive.org/dandiset/001754/draft), version draft. This
version was asked for. It is a repeat attempt: the first attempt, `001754-knierim-2000-01`, read
version 0.260728.1352, and the data owners have since replaced all eight files in the draft with a
new conversion that addresses its findings. The fixes are not yet published, so the draft is read.
A draft can change: the draft read here was last modified on 2026-10-14 at 21:55:09 UTC (the data
check prints the time it has when the notebook runs). The three published versions (0.260227.0029,
0.260227.0031, 0.260728.1352) hold the same blobs; every file of the draft is a new blob, and the
dandiset's description is unchanged. The data check compares the draft with 0.260728.1352 where the
analysis uses the difference.

What the dandiset holds. Three rats on the Neurolab mission (STS-90, April 1998): a preflight
recording of each, flight day 4 (FD4) of each, and flight day 9 (FD9) of Rats 1 and 2. Each file
has sorted units with the original spike files merged into each and their windows, task epochs
(BL baseline, ES Escher staircase, MC Magic Carpet) with their original names, and the 64 x 64 rate
and occupancy maps of the original analysis for ES and MC, each linked to its unit. The flight files
also hold the video position, but only at spike times and only within the rat's own ES and MC
epochs. There is no EEG and no head direction.

Conversion code read. [catalystneuro/mcnaughton2000-to-nwb](https://github.com/catalystneuro/mcnaughton2000-to-nwb)
at commit 734688253c17a32e26d84c57f41fbe48ffbcd009 (2026-10-14), which made the draft's files; the
first attempt read b538d9e. The new commit recovers the original long file names (ES.cell1.rmap,
BL2.cell5) from each tetrode folder's WS_FTP.LOG, links each rate map to its unit, names the epochs
as the originals did, records each unit's spike files and windows, restricts each flight rat's
position to its own ES and MC epochs (merging the shared tracker's samples from both rats' spike
files), stores the lost position and unvisited bins as NaN, leaves out rate maps that are identical
copies of another map, and drops a stray spike from before the window in 48 spike files. The paper's
own analysis code is not linked and was not found.

| Claim | Source | Tested |
|---|---|---|
| `flight-spatial-code`: a robust spatial code on the Escher staircase in flight | Abstract | Yes |
| `three-orthogonal-surfaces`: the code covers each of the three surfaces | Abstract | Blocked: metadata insufficient |
| `si-comparable-to-preflight`: flight spatial information comparable to preflight | Dandiset description | Yes |
| `fd4-abnormal-selectivity`: abnormal selectivity on flight day 4 | Dandiset description | Yes |
| `fig3-stability-across-sessions`: place fields stable across sessions | Figure 3 title | Yes |
| `theta-normal`: normal theta in flight | Dandiset description | Blocked: data missing |
| `ripples-normal`: normal sharp waves and ripples in flight | Dandiset description | Blocked: data missing |

In [ ]:
import json
import re
import time
import warnings
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import remfile
import requests
from scipy import stats

import neurolab as nl

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 300)
warnings.filterwarnings("ignore", category=RuntimeWarning)

OLD_VERSION = "0.260728.1352"  # the version of the earlier attempt, 001754-knierim-2000-01
started = time.time()
sessions = nl.load_all()
by_label = {nl.label(s): s for s in sessions}
print(f"loaded {len(sessions)} files from dandiset {nl.DANDISET_ID} version {nl.DANDISET_VERSION} "
      f"in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output. The status of each
finding of the first attempt on this version is given at the end of the data check.

In [ ]:
api = "https://api.dandiarchive.org/api/dandisets/001754/versions"
listing = requests.get(f"{api}/", timeout=60).json()["results"]
print("Versions and when each was last modified:")
for v in listing:
    print(f"  {v['version']:15s} {v['modified']}")
print("The draft read here was last modified", next(v["modified"] for v in listing if v["version"] == "draft"),
      "; when this attempt was made it was 2026-10-14T21:55:09Z, and a later time means the draft has changed since.")
blobs = {}
for v in [x["version"] for x in listing]:
    assets = requests.get(f"{api}/{v}/assets/", params={"page_size": 100}, timeout=60).json()["results"]
    blobs[v] = sorted((a["path"], a["size"], a.get("blob")) for a in assets)
published = [v for v in blobs if v != "draft"]
print("\nThe published versions hold the same files with the same blobs:", all(blobs[v] == blobs[published[0]] for v in published))
old_blobs = {p: (size, blob) for p, size, blob in blobs[OLD_VERSION]}
print(f"Files of the draft, with their size in {OLD_VERSION} and whether the draft's file is a new blob:")
for path, size, blob in blobs["draft"]:
    print(f"  {path}  {size / 1e6:.2f} MB (was {old_blobs[path][0] / 1e6:.2f} MB)  new blob: {blob != old_blobs[path][1]}")
meta = {v: requests.get(f"{api}/{v}/", timeout=60).json() for v in ["draft", OLD_VERSION]}
print(f"The dandiset description is the same in the draft and {OLD_VERSION}:", meta["draft"]["description"] == meta[OLD_VERSION]["description"])

print("\nEvery group (with its neurodata type) and dataset (shape, type, description) of every file, unfiltered:")
types = {}
for path_url in nl.asset_urls():
    tree = nl.h5_tree(path_url)
    print(f"\n{path_url[0]}: {len(tree)} groups and datasets")
    for line in tree:
        print(" ", line)
        if "  group  " in line and line.split("  group  ")[1]:
            types.setdefault(line.split("  group  ")[1], set()).add(path_url[0])
print("\nNeurodata types of groups, and the number of files holding each:", {t: len(v) for t, v in sorted(types.items())})

The published versions hold the same files; every file of the draft is a new blob, and the
dandiset description has not changed. The listing shows every group and dataset of every file. In
each file `acquisition/`, `analysis/` and the `stimulus/` groups are empty, and the neurodata types
are only those printed under the listing (devices, electrode groups and table, subject, epochs,
units, position and the rate-map table): there is no ElectricalSeries or LFP, so no file holds EEG
or any continuous voltage. The units table now has `tetrode`, `cluster_id`, `spike_times`,
`obs_intervals`, `source_files` and `electrodes`; the epochs table `epoch_name`, `session_type` and
its description; the rate-map table `tetrode`, `source_file`, `archived_file_name`, `epoch_name`,
`cluster_id`, `unit`, `rate_map` and `occupancy_map`; the position one series of x and y in pixels.

In [ ]:
# The full descriptions of the tables and series the analysis uses, from the Rat1 FD9 file (the
# other files carry the same text, except the position description, which names each file's
# partner rat), with the subject's age and description from every file.
path_url = [p for p in nl.asset_urls() if p[0].startswith("sub-Rat1/sub-Rat1_ses-19980425")][0]
with h5py.File(remfile.File(path_url[1]), "r") as f:
    print("experiment_description:", nl._text(f["general/experiment_description"][()]), "\n")
    for name in ["units", "units/spike_times", "intervals/epochs", "processing/ecephys/rate_maps",
                 "processing/ecephys/rate_maps/rate_map", "processing/ecephys/rate_maps/occupancy_map",
                 "processing/behavior/position/spatial_series"]:
        attrs = {k: nl._text(v) for k, v in f[name].attrs.items() if k in ("description", "resolution")}
        print(f"{name}: {attrs}\n")
    print("position reference_frame:", nl._text(f["processing/behavior/position/spatial_series/reference_frame"][()]), "\n")
for p, url in nl.asset_urls():
    with h5py.File(remfile.File(url), "r") as f:
        print(f"{p.split('/')[0]} {p.split('_ses-')[1][:8]}: age {nl._text(f['general/subject/age'][()])!r}; "
              f"{nl._text(f['general/subject/description'][()])}")

The draft's descriptions now state what the first attempt had to find from the values: that
unvisited rate-map bins are NaN (-999 in the source), that the maps are smoothed with an unrecorded
kernel, that the occupancy counts 60 Hz video samples and counts the lost position in bin [63, 0],
how rate-map bins map to pixels, that each tetrode was cluster-cut once so a unit is the same sorted
cluster in every epoch, how the spike times are rounded, that Rat1 and Rat2 shared one tracker on
each flight day, and that the camera view and pixel size are not recorded. Each of these is checked
against the values below. The position description says the trace in a rat's ES and MC epochs "is
taken to be" that rat's position and that the source files do not name the tracked animal. The
age is still "/", now with a subject description saying the age is not recorded. The experiment
description says no EEG was archived; the dandiset's own description still reports EEG.

In [ ]:
# What changed from the version of the earlier attempt. Both versions are read: epochs, units,
# spikes, position and rate maps of each file. In the old files unvisited rate-map bins are -999,
# in the draft NaN; maps are compared with NaN read as -999.
def read_old(path_url):
    with h5py.File(remfile.File(path_url[1]), "r") as f:
        ep, u, rm = f["intervals/epochs"], f["units"], f["processing/ecephys/rate_maps"]
        out = {"epochs": list(zip(ep["start_time"][:], ep["stop_time"][:], nl._column(ep, "session_type"))),
               "units": list(zip(nl._column(u, "tetrode"), u["cluster_id"][:])), "spikes": nl._ragged(u, "spike_times"),
               "maps": rm["rate_map"][:], "occ": rm["occupancy_map"][:], "map_types": nl._column(rm, "session_type"),
               "map_tetrodes": nl._column(rm, "tetrode"), "map_files": nl._column(rm, "source_file")}
        if "processing/behavior" in f:
            ss = f["processing/behavior/position/spatial_series"]
            out["t"], out["xy"] = ss["timestamps"][:], ss["data"][:]
    return out

with ThreadPoolExecutor(8) as pool:
    old_urls = nl.asset_urls(OLD_VERSION)
    old_files = dict(zip([p for p, _ in old_urls], pool.map(read_old, old_urls)))

def overlapping(intervals):
    iv = sorted(intervals)
    return sum(1 for (a1, b1), (a2, b2) in zip(iv, iv[1:]) if a2 < b1)

rows, lost_maps = [], []
for s in sessions:
    o = old_files[s["path"]]
    old_spikes = {(t, int(c)): sp for (t, c), sp in zip(o["units"], o["spikes"])}
    new_spikes = {(t, int(c)): sp for t, c, sp in zip(s["units"].tetrode, s["units"].cluster, s["units"].spikes)}
    same_units = set(old_spikes) == set(new_spikes)
    removed = [len(old_spikes[k]) - len(new_spikes[k]) for k in new_spikes]
    subset = all(np.isin(new_spikes[k], old_spikes[k]).all() for k in new_spikes)
    new_maps = {np.where(np.isnan(m), -999, m).astype(np.float32).tobytes() for m in s["rate_maps"]}
    old_keys = [m.tobytes() for m in o["maps"]]
    groups = Counter(old_keys)
    for k, n in groups.items():
        if k not in new_maps:
            lost_maps.append((nl.label(s), n, [f"{o['map_tetrodes'][i]}/{o['map_files'][i]}" for i, x in enumerate(old_keys) if x == k]))
    row = {"session": nl.label(s), "epochs_old": len(o["epochs"]), "epochs_new": len(s["epochs"]),
           "overlapping_old": overlapping([(a, b) for a, b, _ in o["epochs"]]), "overlapping_new": overlapping(list(zip(s["epochs"].start, s["epochs"].stop))),
           "same_units": same_units, "spikes_removed": int(sum(removed)), "units_losing_spikes": int(sum(r > 0 for r in removed)),
           "new_spikes_subset_of_old": subset, "maps_old": len(o["maps"]), "distinct_maps_old": len(groups), "maps_new": len(s["rate_maps"]),
           "old_distinct_maps_in_new": int(sum(k in new_maps for k in groups))}
    if "t" in s:
        row["position_old"], row["position_new"] = len(o["t"]), len(s["t"])
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))
print("\nOld distinct rate maps that are not in the draft (file, copies in the old file, tetrode/short name of each copy):")
for x in lost_maps:
    print("  ", x)

print("\nPosition. For each draft flight file, its samples whose timestamp is in the old file of the same rat or of the")
print("other rat, and whether the positions agree there (within 2 px, both valid; the old files wrote (0, 255) for a lost position):")
rows = []
for s in sessions:
    if "t" not in s:
        continue
    partner = {"Rat1": "Rat2", "Rat2": "Rat1"}.get(s["subject"])
    row = {"session": nl.label(s), "samples_new": len(s["t"])}
    for who, key in [("same_rat", s["path"]), ("other_rat", by_label[f"{partner} {s['day']}"]["path"] if partner else None)]:
        if key is None:
            continue
        o = old_files[key]
        common, i_new, i_old = np.intersect1d(s["t"], o["t"], return_indices=True)
        old_valid = ~((o["xy"][i_old, 0] == 0) & (o["xy"][i_old, 1] == 255))
        both = s["valid"][i_new] & old_valid
        dist = np.hypot(*(s["xy"][i_new][both] - o["xy"][i_old][both]).T)
        row[f"in_old_{who}_file"] = round(len(common) / len(s["t"]), 3)
        row[f"{who}_within_2px"] = round(float((dist < 2).mean()), 3)
        row[f"{who}_lost_agrees"] = round(float((~s["valid"][i_new] == ~old_valid).mean()), 3)
    in_either = np.isin(s["t"], np.concatenate([old_files[s["path"]]["t"]] + ([old_files[by_label[f"{partner} {s['day']}"]["path"]]["t"]] if partner else [])))
    row["in_neither_old_file"] = int((~in_either).sum())
    row["their_time_range_s"] = f"{s['t'][~in_either].min():.0f}-{s['t'][~in_either].max():.0f}" if (~in_either).any() else ""
    row["old_same_rat_samples_not_in_new"] = int((~np.isin(old_files[s["path"]]["t"], s["t"])).sum())
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))

The draft holds the same units as 0.260728.1352, and every spike of the draft is a spike of the
earlier version: 1 to 16 spikes per file were removed, the stray first spikes the units' description
mentions. Rat1 FD4 has five epochs in place of six, without the overlapping baseline. The rate maps
of the draft are all among the earlier version's maps; what is missing are the six groups of
identical maps, every copy of which is left out (14 maps), so these six maps are no longer in the
dandiset. The draft's position samples are all at timestamps of the earlier files of the same rat or
of the other rat, where they agree (99.6 to 100% within 2 pixels), except 44 in the last 7 s of Rat2
FD9 and 1259 in the last 40 s of Rat3 FD4, both within MC epochs. Each file now holds the samples of
both rats' spike files within its own rat's task epochs and none elsewhere.

In [ ]:
rows = []
for s in sessions:
    ep = s["epochs"]
    rows.append({"file": s["path"].split("/")[1][:31], "rat": s["subject"], "day": s["day"], "start": s["start"][:16],
                 "units": len(s["units"]), "spikes": int(sum(len(x) for x in s["units"]["spikes"])),
                 "tetrodes": s["units"]["tetrode"].nunique(),
                 "epochs": " ".join(f"{n}[{a:.0f}-{b:.0f}]" for a, b, n in ep[["start", "stop", "name"]].itertuples(index=False)),
                 "rate_maps": len(s["maps"]), "map_epochs": "/".join(sorted(s["maps"]["epoch_name"].unique())),
                 "position_samples": len(s["t"]) if "t" in s else 0})
info = pd.DataFrame(rows)
print(info.to_string(index=False))
print("\nFiles per rat and day:"); print(pd.crosstab(info.rat, info.day).to_string())
for s in sessions:
    print(f"{nl.label(s):14s} {s['description']}")
print("\nTetrodes with units, per file:")
for s in sessions:
    print(f"  {nl.label(s):14s}", sorted(s["units"].tetrode.unique(), key=lambda t: int(t[2:])))

Three rats, eight files: preflight and FD4 for each rat, FD9 for Rats 1 and 2, as the description
says. Rate maps exist only for ES and MC epochs, not for baseline; Rat3 FD4 has two ES epochs, ES1
and ES2, each with its own maps. Position exists only in the five flight files. No epochs overlap
now. Rat2's preflight units are on tetrodes TT0 to TT10 and its flight units on TT12 to TT22, with no
tetrode in common, while Rat1's and Rat3's preflight and flight tetrodes overlap. The conversion
notes list whether the preflight folders are assigned to the right rats as an open question;
nothing in the files settles it, so the per-rat comparisons with preflight below assume the
assignment.

In [ ]:
print("Position, flight files. Stated unit:", {s['xy_unit'] for s in sessions if 'xy' in s})
rows = []
for s in sessions:
    if "xy" not in s:
        continue
    xy, v, t = s["xy"], s["valid"], s["t"]
    own = np.unique(np.concatenate(list(s["units"]["spikes"])))
    partner = {"Rat1": "Rat2", "Rat2": "Rat1"}.get(s["subject"])
    other = np.unique(np.concatenate(list(by_label[f"{partner} {s['day']}"]["units"]["spikes"]))) if partner else np.array([])
    task = np.zeros(len(t), bool)
    own_task = np.zeros(len(own), bool)
    for a, b, typ in s["epochs"][["start", "stop", "type"]].itertuples(index=False):
        if typ != "BL":
            task |= (t >= a) & (t <= b)
            own_task |= (own >= a) & (own <= b)
    rows.append({"session": nl.label(s), "samples": len(t), "lost_(NaN)": round(1 - v.mean(), 3),
                 "x_min": xy[v, 0].min(), "x_max": xy[v, 0].max(), "y_min": xy[v, 1].min(), "y_max": xy[v, 1].max(),
                 "t_first": t[0], "t_last": t[-1], "in_own_ES_MC_epochs": round(task.mean(), 3),
                 "at_own_spike_times": round(float(np.isin(t, own).mean()), 3),
                 "at_other_rat_spike_times": round(float(np.isin(t, other).mean()), 3) if partner else np.nan,
                 "at_either": round(float(np.isin(t, np.union1d(own, other)).mean()), 3),
                 "own_task_spike_times_with_sample": round(float(np.isin(own[own_task], t).mean()), 4)})
print(pd.DataFrame(rows).round(2).to_string(index=False))

Position is in video pixels from 0 to 255 in x and y. Lost positions are NaN (6 to 13% of
samples); these are dropped. Every sample lies in one of the file's own ES or MC epochs and at a
spike time of one of the two rats sharing the tracker (in Rat3's file, of its own units); in Rat2's
files most samples are at Rat1's spike times (0.56 on FD4, 0.79 on FD9). Every spike of the file's
own units in its task epochs has a sample. The trace is sampled only when some unit fires, at
irregular intervals, consistent with the description. No occupancy time can be built from it; the
occupancy maps of the original analysis are used.

In [ ]:
# The position lookup on synthetic input: samples at 1.0 s, 1.1 s (the lost position, NaN) and 1.5 s.
fake = {"t": np.array([1.0, 1.1, 1.5]), "xy": np.array([[40.0, 40.0], [np.nan, np.nan], [200.0, 200.0]])}
fake["valid"] = ~np.isnan(fake["xy"]).any(axis=1)
found, r, c = nl.PositionLookup(fake)(np.array([1.0, 1.1, 1.12, 1.4, 2.0]))
print("times 1.0, 1.1, 1.12, 1.4, 2.0 s; found:", found.tolist(), "; rows:", r.tolist(), "; columns:", c.tolist())
assert found.tolist() == [True, False, False, True, False] and r.tolist() == [10, 50] and c.tolist() == [10, 50]

The lookup gives a spike the position of the nearest sample within 0.25 s and drops the spike
when that sample is the lost position (NaN in the draft). On the synthetic input, the spikes at 1.1 s
(the lost sample itself) and at 1.12 s (nearest to it) are dropped, as is the spike at 2.0 s, 0.5 s
from any sample.

In [ ]:
print("Do the Rat1 and Rat2 files of a flight day share position samples, and which epochs do their samples fall in?")
for day in ["FD4", "FD9"]:
    a, b = by_label[f"Rat1 {day}"], by_label[f"Rat2 {day}"]
    print(f"  {day}: timestamps in both files: {len(np.intersect1d(a['t'], b['t']))}")
    for s in (a, b):
        n = {f"{name} ({typ})": int(((s["t"] >= x) & (s["t"] <= y)).sum()) for x, y, typ, name in s["epochs"][["start", "stop", "type", "name"]].itertuples(index=False)}
        print(f"    {nl.label(s)} samples per own epoch: {n}")
    print("    Rat1 epochs:", list(a["epochs"][["start", "stop", "name"]].itertuples(index=False, name=None)))
    print("    Rat2 epochs:", list(b["epochs"][["start", "stop", "name"]].itertuples(index=False, name=None)))

The Rat1 and Rat2 files of a flight day no longer share any timestamp: each holds samples only in
its own rat's ES and MC epochs, and the epochs show that the rats took turns (when one rat is
labelled ES or MC, the other is labelled BL). In 0.260728.1352 both files held the same trace over
the whole session, presented as each rat's position; the draft assigns each window to one rat. The
next checks ask whether the trace in each window is the position of the rat it is assigned to.

In [ ]:
print("Distinct occupancy maps per epoch name, per file:", {nl.label(s): dict(Counter(n for n, _, _ in nl.map_occupancy_groups(s))) for s in sessions})
rows = []
for s in sessions:
    if "xy" not in s:
        continue
    for a, b, typ, name in s["epochs"][["start", "stop", "type", "name"]].itertuples(index=False):
        if typ == "BL":
            continue
        scores = nl.occupancy_match(s, a, b)
        occ = nl.epoch_occupancy(s, name)
        rows.append({"session": nl.label(s), "epoch": f"{name} {a:.0f}-{b:.0f}", "duration_s": b - a,
                     "match_own_name": round(scores[name], 2), "best_match": max(scores, key=scores.get),
                     "match_other_names": {k: round(v, 2) for k, v in scores.items() if k != name},
                     "occupancy_counts": int(occ.sum()), "counts_per_s": round(occ.sum() / (b - a), 1)})
print("Correlation of each binary occupancy map with the binary histogram of the epoch's position samples:")
print(pd.DataFrame(rows).to_string(index=False))

Each epoch name has one occupancy map in its file. For each ES and MC epoch, the position samples
in the epoch cover the bins of the occupancy map of the same name best (0.70 to 0.88, against 0.30
to 0.67 for the other maps), including Rat3's ES1 and ES2. The occupancy totals are 38 to 57 counts
per second of the epoch, the same order in every epoch and below the 60 Hz video rate the occupancy
description now states; why not every frame is counted is not stated. The epoch names, times and position are therefore consistent with the
original analysis. This is a check of position against occupancy; whether spikes and position share
a clock is shown with spike-based maps below.

In [ ]:
print("Rate-map values:")
r_all = np.concatenate([s["rate_maps"].ravel() for s in sessions])
o_all = np.concatenate([s["occupancy_maps"].ravel() for s in sessions])
print("  NaN bins:", round(float(np.isnan(r_all).mean()), 3), "; NaN exactly where occupancy is 0:", bool((np.isnan(r_all) == (o_all == 0)).all()),
      "; negative values:", int((r_all < 0).sum()))
print("  max rate per file (Hz):", {nl.label(s): round(float(np.nanmax(s["rate_maps"])), 1) for s in sessions})
print("\nOccupancy in bin [63, 0], the bin of the lost position (0, 255), as a fraction of each occupancy map:")
for s in sessions:
    print("  ", nl.label(s), [f"{name} {occ[nl.SENTINEL_BIN] / occ.sum():.3f}" for name, occ, _ in nl.map_occupancy_groups(s)])

maps = nl.rma_table(sessions)
print("\nRate maps per file and type: all, identical copies of another map in the same file, archived short names with CEL:")
print(maps.groupby(["rat", "day", "type"]).agg(maps=("info", "size"), duplicates=("duplicate", "sum"), cel_names=("cel_name", "sum")).to_string())
print("\nExamples of source_file and archived_file_name:", [f"{a} = {b}" for a, b in zip(sessions[2]["maps"].source_file[:8], sessions[2]["maps"].archived_file_name[:8])])
print("source_file names that do not read <epoch name>.cell<cluster id>.rmap:",
      sum(f != f"{n}.cell{c}.rmap" for s in sessions for f, n, c in zip(s["maps"].source_file, s["maps"].epoch_name, s["maps"].cluster)))
print("Rate maps whose unit has a different tetrode or cluster id:",
      sum((s["units"].tetrode[u] != t) or (s["units"].cluster[u] != c) for s in sessions for u, t, c in zip(s["maps"].unit, s["maps"].tetrode, s["maps"].cluster)))
print("Rate maps whose unit has no spike file of the map's epoch:",
      sum(f"{t}/{n}.cell{c}" not in s["units"].source_files[u] for s in sessions for u, t, n, c in zip(s["maps"].unit, s["maps"].tetrode, s["maps"].epoch_name, s["maps"].cluster)))
pk = []
for s in sessions:
    for i in range(len(s["maps"])):
        occ = nl.clean_occupancy(s["occupancy_maps"][i])
        rate = np.where((occ > 0) & np.isfinite(s["rate_maps"][i]), s["rate_maps"][i], -1.0)
        j = np.unravel_index(rate.argmax(), rate.shape)
        pk.append({"session": nl.label(s), "peak_rate": rate[j], "occupancy_at_peak": occ[j]})
pk = pd.DataFrame(pk)
print("\nOccupancy counts in the bin of each map's peak rate:")
print(pk.groupby("session").agg(maps=("peak_rate", "size"), max_peak_hz=("peak_rate", "max"), median_occupancy_at_peak=("occupancy_at_peak", "median"),
      peaks_in_bins_below_5=("occupancy_at_peak", lambda o: int((o < nl.MIN_OCC).sum()))).round(1).to_string())
print("Five highest peaks:"); print(pk.nlargest(5, "peak_rate").round(1).to_string(index=False))

Unvisited rate-map bins are NaN, exactly where the occupancy is zero, as the description now
says, and no value is negative. The lost position is still counted in the original occupancy (bin
[63, 0], up to 19% of an occupancy map in flight, under 2% preflight), as the occupancy description
now says; that bin is left out here. No file holds two identical rate maps. Every `source_file` reads
`<epoch name>.cell<cluster id>.rmap`, every map's unit has the map's tetrode and cluster id, and
every map's unit has a spike file of the map's epoch; the old 8.3 short names are kept in
`archived_file_name`. The "CEL names only" variant below uses those short names, as the first
attempt did.

The highest peak rate, 277.8 Hz in a Rat1 FD9 map, lies in a bin with 55 occupancy counts, not a
sparsely visited bin. Per file, 9 to 36 maps have their peak in a bin with fewer than 5 counts;
spatial information over bins with at least 5 counts is shown as a variant of the rate-map claims.

In [ ]:
# What apparatus the preflight epochs labelled ES and MC were recorded on: the dandiset description,
# and the extent of every distinct occupancy map (visited bins, rows and columns of the 64 x 64 grid).
desc = meta["draft"]["description"]
print("Dandiset description, sentences on the track:")
for sentence in desc.split(". "):
    if "track" in sentence:
        print("  ", sentence.strip())
print("\nEpoch names per file:", {nl.label(s): sorted(set(s["epochs"].name)) for s in sessions})
rows, footprint = [], {}
for s in sessions:
    for name, occ, idx in nl.map_occupancy_groups(s):
        v = nl.clean_occupancy(occ) > 0
        footprint.setdefault((s["subject"], s["day"], nl.epoch_type(name)), v)
        rr, cc = np.where(v)
        rows.append({"session": nl.label(s), "epoch": name, "maps": len(idx), "visited_bins": int(v.sum()),
                     "rows": f"{rr.min()}-{rr.max()}", "columns": f"{cc.min()}-{cc.max()}", "occupancy_counts": int(occ.sum())})
print(pd.DataFrame(rows).to_string(index=False))
jaccard = lambda a, b: round(float((a & b).sum() / (a | b).sum()), 2)
rows = []
for rat in ["Rat1", "Rat2", "Rat3"]:
    fp = lambda day, typ: footprint[(rat, day, typ)]
    for day in ["FD4", "FD9"]:
        if (rat, day, "ES") in footprint:
            rows.append({"rat": rat, "flight_day": day, "preflight ES / flight ES": jaccard(fp("preflight", "ES"), fp(day, "ES")),
                         "preflight MC / flight MC": jaccard(fp("preflight", "MC"), fp(day, "MC")),
                         "preflight ES / preflight MC": jaccard(fp("preflight", "ES"), fp("preflight", "MC")),
                         "flight ES / flight MC": jaccard(fp(day, "ES"), fp(day, "MC"))})
print("\nOverlap of visited bins (Jaccard index; for Rat3 FD4 the ES1 map):")
print(pd.DataFrame(rows).to_string(index=False))

The dandiset's description says the preflight recordings were made on a flat rectangular track,
but the preflight files label their epochs ES and MC and hold ES and MC rate maps, as the flight
files do; the draft's files describe MC as a flat two-dimensional track and say nothing of a
different apparatus preflight. The preflight occupancy maps cover 827 to 1446 bins within rows 20 to
63 and columns 4 to 56; the flight maps reach row 15 or 16 and cover 1121 to 2662 bins. The visited
bins of preflight ES and flight ES overlap with a Jaccard index of 0.27 to 0.49, below the overlap
of flight ES with flight MC (0.42 to 0.67). The camera view on the ground and in the shuttle may
differ, so the footprints cannot show which apparatus was used. What the preflight epochs labelled
ES and MC were recorded on is not established; if the description is right, the comparisons with
preflight below compare the Escher staircase in flight with another apparatus on the ground.

In [ ]:
# For each flight unit and task epoch, the correlation of the unit's spike-count map with the
# spike-count prediction (rate x occupancy) of the rate map the dandiset links to it, and of every
# other map of the same tetrode and epoch.
rows = []
for s in sessions:
    if "xy" not in s:
        continue
    lookup = nl.PositionLookup(s)
    for a, b, typ, name in s["epochs"][["start", "stop", "type", "name"]].itertuples(index=False):
        if typ == "BL":
            continue
        for ui, u in s["units"].iterrows():
            sp = u.spikes[(u.spikes >= a) & (u.spikes <= b)]
            found, r, c = lookup(sp)
            if found.sum() < nl.MIN_SPIKES:
                continue
            counts = nl.count_map(r, c)
            cand = s["maps"][(s["maps"].tetrode == u.tetrode) & (s["maps"].epoch_name == name)]
            pred = {i: np.nan_to_num(s["rate_maps"][i]) * nl.clean_occupancy(s["occupancy_maps"][i]) for i in cand.index}
            rs = {i: np.corrcoef(p.ravel(), counts.ravel())[0, 1] for i, p in pred.items()}
            linked = list(cand.index[cand.unit == ui])
            row = {"session": nl.label(s), "type": typ, "epoch": name, "tetrode": u.tetrode, "cluster": u.cluster, "spikes": int(found.sum()),
                   "maps_on_tetrode": len(cand), "has_linked_map": bool(linked)}
            if linked:
                i = linked[0]
                rate_i, occ_i = s["rate_maps"][i], nl.clean_occupancy(s["occupancy_maps"][i])
                positive = (np.nan_to_num(rate_i) > 0) & (occ_i > 0)
                row |= {"r_linked": rs[i], "linked_is_best": max(rs, key=rs.get) == i,
                        "r_linked_x_y_swapped": np.corrcoef(pred[i].ravel(), counts.T.ravel())[0, 1],
                        "best_other_r": max([v for k, v in rs.items() if k != i], default=np.nan),
                        "positive_bins_without_spikes": float((positive & (counts == 0)).sum() / positive.sum()),
                        "rate_x_occupancy_per_spike": float(pred[i].sum() / found.sum())}
            rows.append(row)
link = pd.DataFrame(rows)
print(f"Unit-epochs with at least {nl.MIN_SPIKES} spikes at a known position: {len(link)}; with a linked rate map: {int(link.has_linked_map.sum())}")
lk = link[link.has_linked_map]
print(lk.groupby(["session", "epoch"]).agg(n=("r_linked", "size"), median_r_linked=("r_linked", "median"), median_r_swapped=("r_linked_x_y_swapped", "median"),
      median_best_other_r=("best_other_r", "median"), linked_is_best=("linked_is_best", "mean")).round(2).to_string())
print(f"\nAll: median r with the linked map {lk.r_linked.median():.2f}; the linked map is the best of its tetrode and epoch in "
      f"{int(lk.linked_is_best.sum())} of {len(lk)}; r > 0.5 in {(lk.r_linked > 0.5).sum()}")
print("Unit-epochs without a linked map:"); print(link[~link.has_linked_map][["session", "epoch", "tetrode", "cluster", "spikes", "maps_on_tetrode"]].to_string(index=False))
good = lk[lk.r_linked > 0.5]
print(f"\nLinked maps with r > 0.5 ({len(good)}): fraction of bins with a positive rate where the unit has no spike, median "
      f"{good.positive_bins_without_spikes.median():.2f}; rate x occupancy per spike, median "
      f"{good.rate_x_occupancy_per_spike.median():.1f} (range {good.rate_x_occupancy_per_spike.min():.1f} to "
      f"{good.rate_x_occupancy_per_spike.max():.1f})")
print(good.groupby("session").rate_x_occupancy_per_spike.median().round(1).to_string())

With position binned as x to columns and y to rows at 4 pixels per bin, as the rate-map table now
states, each of the 211 unit-epochs with 50 or more spikes at a known position has a linked rate
map, and that map is the best match among the maps of its tetrode and epoch for 201 of them (median
r 0.57 against a median of 0.13 to 0.41 for the best other map; -0.01 to 0.07 with x and y swapped).
So the rate maps are now linked to their units, as the table says, and were made from the same
spikes and positions. In 0.260728.1352 the `cell_number` column matched the cluster id for 84 of
211 unit-epochs.

For the 135 linked maps that correlate above 0.5, in the median map 91% of the bins with a positive
rate hold no spike of the unit, so the maps are smoothed, as the description now says. Their summed
rate x occupancy is about 22 per spike of the unit (median per file 21.1 to 23.8), so below it is
used as a quantity proportional to a map's spike count. The preflight maps cannot be checked this
way, as the preflight files have no position.

In [ ]:
# Spike times: sorting, resolution and zero intervals by time range (the units' description says
# times have six significant digits), spikes outside each unit's obs_intervals, and the unit
# windows that differ from their epoch's.
ranges = [(0, 1000), (1000, 10000), (10000, np.inf)]
rows = []
for s in sessions:
    allsp = np.concatenate(list(s["units"].spikes))
    for lo, hi in ranges:
        z = n = 0
        for sp in s["units"].spikes:
            w = sp[(sp >= lo) & (sp < hi)]
            n += max(len(w) - 1, 0)
            z += int((np.diff(w) == 0).sum())
        distinct = np.unique(allsp[(allsp >= lo) & (allsp < hi)])
        if len(distinct) > 1:
            rows.append({"session": nl.label(s), "time_range_s": f"{lo}-{hi}", "spikes": int(((allsp >= lo) & (allsp < hi)).sum()),
                         "smallest_step_s": round(float(np.diff(distinct).min()), 4), "intervals": n, "zero_intervals": z,
                         "zero_fraction": round(z / max(n, 1), 3)})
print("Spike times are sorted in every unit:", all((np.diff(sp) >= 0).all() for s in sessions for sp in s["units"].spikes))
print(pd.DataFrame(rows).to_string(index=False))

rows = []
for s in sessions:
    ep = {n: (a, b) for a, b, n in s["epochs"][["start", "stop", "name"]].itertuples(index=False)}
    outside = differ = 0
    for _, u in s["units"].iterrows():
        inside = np.zeros(len(u.spikes), bool)
        for (a, b), f in zip(u.obs_intervals, u.source_files):
            inside |= (u.spikes >= a) & (u.spikes <= b)
            differ += ep[f.split("/")[1].split(".")[0]] != (a, b)
        outside += int((~inside).sum())
    iv = sorted(zip(s["epochs"].start, s["epochs"].stop))
    rows.append({"session": nl.label(s), "units": len(s["units"]), "spike_files": int(s["units"].source_files.map(len).sum()),
                 "spikes_outside_obs_intervals": outside, "unit_windows_differing_from_epoch": differ,
                 "overlapping_epochs": sum(1 for (a1, b1), (a2, b2) in zip(iv, iv[1:]) if a2 < b1),
                 "median_epochs_per_unit": float(s["units"].source_files.map(len).median())})
print("\nUnits, their spike files and windows:")
print(pd.DataFrame(rows).to_string(index=False))
s = by_label["Rat1 FD4"]
print("Rat1 FD4 unit windows in BL1 that differ from the epoch (2525-3390):",
      Counter(tuple(w) for _, u in s["units"].iterrows() for w, f in zip(u.obs_intervals, u.source_files) if "/BL1." in f and tuple(w) != (2525.0, 3390.0)))

Spike times are sorted. Their resolution follows the time, as the units' description says: 1 ms or
finer below 1000 s, 10 ms from 1000 s and 0.1 s from 10000 s. Zero intervals within a unit follow
the resolution: none or nearly none below 1000 s (0 to 3), 5 to 15% at 10 ms, and 35 to 53% at 0.1 s.
This explains the zero intervals the first attempt could not account for (Rat1's preflight MC epoch
crosses 1000 s). Within FD9 sessions, spikes after 10000 s (the last 151 s of each) are resolved
to 0.1 s only; of the task epochs only Rat2's FD9 MC epoch reaches it, and no claim is judged on it. No spike lies outside its unit's `obs_intervals`. Units
merge the per-epoch spike files of one cluster; 14 Rat1 FD4 units have their BL1 window at 2720 to
3390 s, inside the BL1 epoch (2525 to 3390 s), as the epochs' description explains. Every result here
except the stability claim is computed per epoch; the stability claim uses that a unit is the same
sorted cluster in both epochs, which the units' description now states.

In [ ]:
# Clock and attribution contrast: spatial information of each rat's units in its own ES epoch and
# in the other rat's ES epoch of the same day. In each window the position comes from the file of
# the rat labelled as on the task, whose samples merge both rats' spike times, with that file's
# occupancy map for the epoch.
rng_check = np.random.default_rng(1754)
rows = []
for day in ["FD4", "FD9"]:
    files = {rat: by_label[f"{rat} {day}"] for rat in ["Rat1", "Rat2"]}
    for rat, other in [("Rat1", "Rat2"), ("Rat2", "Rat1")]:
        for owner in (rat, other):
            src = files[owner]
            for a, b, typ, name in src["epochs"][["start", "stop", "type", "name"]].itertuples(index=False):
                if typ != "ES":
                    continue
                df = nl.epoch_units(files[rat], a, b, nl.epoch_occupancy(src, name), rng_check, n_shuffles=200, position=src)
                d = df.dropna(subset=["info"])
                rows.append({"day": day, "units_of": rat, "ES_epoch_of": owner, "window": f"{a:.0f}-{b:.0f}",
                             "units": len(d), "significant": int((d.p < 0.05).sum()),
                             "fraction": round(float((d.p < 0.05).mean()), 2) if len(d) else np.nan,
                             "binom_p_vs_0.05": stats.binomtest(int((d.p < 0.05).sum()), len(d), 0.05, alternative="greater").pvalue if len(d) else np.nan,
                             "median_info": round(float(d["info"].median()), 3) if len(d) else np.nan})
contrast = pd.DataFrame(rows)
print(contrast.to_string(index=False))

This contrast tests whether spikes and position share a clock and whose position the trace is.
Each rat's units are tested (200 shuffles) in its own ES epoch and in the other rat's ES epoch of
the same day, with the trace and occupancy of the file of the rat labelled as on the task. Rat1's
units are selective in their own ES epochs (11 of 20 on FD4, 10 of 25 on FD9, against 5% expected by
chance), so for Rat1 spikes and position share a clock. Rat2's units are no more selective than
chance in their own ES epochs (1 of 16 on FD4, 2 of 28 on FD9) or in Rat1's (0 of 15, 0 of 41).
Rat1's units are selective in Rat2's FD4 ES epoch (10 of 15), when Rat1 is labelled as on baseline,
but not in Rat2's FD9 ES epoch (1 of 36). So, as in 0.260728.1352, a shared clock could not be
established for Rat2's spikes and the position, and in Rat2's FD4 ES epoch the trace is consistent
with Rat1's position, not Rat2's.

In [ ]:
# Whose position is the trace in each task window? For every ES and MC window, the samples of the
# file of the rat on the task, the fraction of them at each rat's spike times, and each rat's
# spikes per second in the window.
rows = []
for day in ["FD4", "FD9"]:
    f1, f2 = by_label[f"Rat1 {day}"], by_label[f"Rat2 {day}"]
    spikes = {"Rat1": np.unique(np.concatenate(list(f1["units"].spikes))), "Rat2": np.unique(np.concatenate(list(f2["units"].spikes)))}
    for owner, src in [("Rat1", f1), ("Rat2", f2)]:
        for a, b, typ, name in src["epochs"][["start", "stop", "type", "name"]].itertuples(index=False):
            if typ == "BL":
                continue
            w = (src["t"] >= a) & (src["t"] <= b)
            tt = src["t"][w]
            other_file = f2 if owner == "Rat1" else f1
            row = {"day": day, "window": f"{owner} {name} {a:.0f}-{b:.0f}", "samples_per_s": round(w.sum() / (b - a), 1),
                   "other_file_samples": int(((other_file["t"] >= a) & (other_file["t"] <= b)).sum()),
                   "lost_fraction": round(float(1 - src["valid"][w].mean()), 3)}
            for rat in ["Rat1", "Rat2"]:
                row[f"at_{rat}_spikes"] = round(float(np.isin(tt, spikes[rat]).mean()), 3)
                row[f"{rat}_spikes_per_s"] = round(((spikes[rat] >= a) & (spikes[rat] <= b)).sum() / (b - a), 1)
            rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# Movement of the trace in each task window (Rat3's for reference, as it had its own tracker):
# steps between consecutive valid samples, path length per second of the window, and the share of
# steps over 20 pixels. Then each of Rat1's and Rat2's units tested against the trace in its own and
# the other rat's MC windows, as for the ES windows above (200 shuffles).
rows = []
for s in sessions:
    if "xy" not in s:
        continue
    for a, b, typ, name in s["epochs"][["start", "stop", "type", "name"]].itertuples(index=False):
        if typ == "BL":
            continue
        w = (s["t"] >= a) & (s["t"] <= b) & s["valid"]
        step = np.hypot(*np.diff(s["xy"][w], axis=0).T)
        dt = np.diff(s["t"][w])
        rows.append({"window": f"{nl.label(s)} {name} {a:.0f}-{b:.0f}", "valid_samples": int(w.sum()),
                     "path_px_per_s": round(float(step.sum() / (b - a)), 1), "median_step_px": round(float(np.median(step)), 2),
                     "steps_over_20px": round(float((step > 20).mean()), 4),
                     "px_per_s_over_gaps_0.5s_or_more": round(float(step[dt >= 0.5].sum() / dt[dt >= 0.5].sum()), 1) if (dt >= 0.5).any() else np.nan})
print(pd.DataFrame(rows).to_string(index=False))

rng_mc = np.random.default_rng(1755)
rows = []
for day in ["FD4", "FD9"]:
    files = {rat: by_label[f"{rat} {day}"] for rat in ["Rat1", "Rat2"]}
    for rat, other in [("Rat1", "Rat2"), ("Rat2", "Rat1")]:
        for owner in (rat, other):
            src = files[owner]
            for a, b, typ, name in src["epochs"][["start", "stop", "type", "name"]].itertuples(index=False):
                if typ != "MC":
                    continue
                d = nl.epoch_units(files[rat], a, b, nl.epoch_occupancy(src, name), rng_mc, n_shuffles=200, position=src).dropna(subset=["info"])
                k = int((d.p < 0.05).sum())
                rows.append({"day": day, "units_of": rat, "MC_epoch_of": owner, "window": f"{a:.0f}-{b:.0f}", "units": len(d), "significant": k,
                             "fraction": round(k / len(d), 2) if len(d) else np.nan,
                             "binom_p_vs_0.05": stats.binomtest(k, len(d), 0.05, alternative="greater").pvalue if len(d) else np.nan})
print("\nEach rat's units against the trace in its own and the other rat's MC windows:")
print(pd.DataFrame(rows).to_string(index=False))

Within each task window only the file of the rat on the task has samples, and they fall at both
rats' spike times. Rat1's units fire more in Rat1's FD9 task windows (35.7 and 47.9 spikes per
second in ES and MC, against 19.0 and 21.8 in Rat2's), while Rat2's units fire at about the same
rate in Rat2's windows as in Rat1's (on FD9, 7.7 and 3.8 against 8.0 and 4.5; on FD4, 10.0 and 6.0
against 7.7 and 5.2). The files do not name the animal the tracker followed, and the draft says so;
whose position the trace is in Rat2's task windows could not be established. The trace moves as a
tracked animal would in every window: the median step between samples is 0.50 to 1.18 pixels, 0.6
to 3.4% of steps exceed 20 pixels, and the path length is 20.5 to 91.6 pixels per second (33.0 in
Rat2's FD9 ES window, 63.7 in Rat1's, 28.1 and 20.5 in Rat3's two ES windows). It is a continuous
track in Rat2's windows, but the movement does not show whose. In the MC windows neither rat's
units are selective, in their own or the other rat's windows (0 to 2 of 11 to 32 units), so the MC
windows cannot settle it either. The judged FD9
session of Rat2 is kept, as the criterion requires, with this caveat.

Findings of `001754-knierim-2000-01` on the draft, from the outputs above:

- The Rat1 and Rat2 files held the same trace, presented as each rat's position. Partly fixed. Each
  file now holds the trace only in its own rat's task epochs (no timestamp in common), and the
  position description states the shared tracker, the assumption that the trace in a rat's task
  epochs is that rat's, and that the files do not name the tracked animal. Rat2's units are still
  not selective in any window, and Rat1's are selective in Rat2's FD4 ES window (10 of 15), so whose
  position the trace is in Rat2's epochs is still not established.
- Position only at spike times, none preflight. Still present; the description now states it.
- The description places the preflight recordings on a flat rectangular track, while the files
  label them ES and MC. Still present; the dandiset description is unchanged.
- `cell_number` was not a cell number, so rate maps could not be linked to units. Fixed: every map
  links to a unit with its tetrode and cluster, and the linked map is the best match for 201 of 211
  unit-epochs.
- The rate maps are smoothed with an unstated kernel. Partly fixed: the description states that
  they are smoothed and that the kernel is not recorded.
- -999 for unvisited bins and the lost position counted in bin [63, 0], both undocumented. Fixed:
  unvisited bins are NaN, lost positions are NaN, and the occupancy description states the bin
  [63, 0] count.
- Identical copies of rate maps, some under another tetrode. Fixed: no file holds two identical
  maps. The draft leaves out every copy in the six groups (14 maps), so those six maps, whose unit
  is ambiguous, are no longer in the dandiset.
- Units merge per-epoch spike files without saying a cluster id names one neuron across epochs, and
  lack the spike-file columns. Fixed: the units' description says each tetrode was cut once, and
  `source_files` and `obs_intervals` give each unit's files and windows.
- No EEG, though the description reports theta and ripples. Partly fixed: the files' experiment
  description says no EEG was archived; the dandiset description still reports EEG.
- Spike times at 10 ms with unexplained zero intervals, and two overlapping baseline epochs in Rat1
  FD4. Fixed: the resolution is documented and the zero intervals follow it, and the epochs no
  longer overlap.
- The age is "/" and the pixel scale and camera view are not described. Partly fixed: the subject
  description and the position's reference frame now say the age, the camera view and the pixel
  size are not recorded, and the map geometry is given in pixels; the values themselves are still
  missing.

## flight-spatial-code

> Hippocampal neurons recorded on the space shuttle form a robust spatial code on the
> three-dimensional Escher staircase track in weightlessness. (Abstract)

Criterion (from the claims file): per in-flight ES epoch, a unit is significant when its spatial
information exceeds the 95th percentile of 500 circular-shift shuffles. Judged on the two FD9
sessions: reproduced when the fraction of significant units is at least 0.5 in both.

In [ ]:
rng_c1 = np.random.default_rng(20000301)
results = []
for s in sessions:
    if "xy" not in s:
        continue
    for a, b, typ, name in s["epochs"][["start", "stop", "type", "name"]].itertuples(index=False):
        if typ in ("ES", "MC"):
            df = nl.epoch_units(s, a, b, nl.epoch_occupancy(s, name), rng_c1)
            df["type"] = typ
            df["epoch"] = f"{typ} {a:.0f}-{b:.0f}"
            results.append(df)
units = pd.concat(results, ignore_index=True)
tested = units.dropna(subset=["info"]).copy()
tested["sig"] = tested.p < 0.05

def summary(d):
    n, k = len(d), int(d.sig.sum())
    return pd.Series({"units_in_file": np.nan, "tested": n, "significant": k, "fraction": round(k / n, 3) if n else np.nan,
                      "binom_p_vs_0.05": stats.binomtest(k, n, 0.05, alternative="greater").pvalue if n else np.nan,
                      "median_info": round(d["info"].median(), 3), "median_null95": round(d["null95"].median(), 3)})

per = tested.groupby(["type", "rat", "day", "epoch"]).apply(summary)
per["units_in_file"] = units.groupby(["type", "rat", "day", "epoch"]).size()
per["dropped_<50_spikes"] = per["units_in_file"] - per["tested"]
print(per.to_string())

In [ ]:
es = tested[tested.type == "ES"]
print("Judged sessions (FD9 ES):")
for rat in ["Rat1", "Rat2"]:
    d = es[(es.rat == rat) & (es.day == "FD9")]
    print(f"  {rat} FD9: {d.sig.sum()} of {len(d)} significant, fraction {d.sig.mean():.3f}, "
          f"binomial p {stats.binomtest(int(d.sig.sum()), len(d), 0.05, alternative='greater').pvalue:.2e}")
print("\nThreshold variants, fraction of significant ES units by session:")
var = []
for (rat, day, epoch), d in es.groupby(["rat", "day", "epoch"]):
    row = {"session": f"{rat} {day} {epoch}", "n": len(d)}
    for alpha in [0.05, 0.01]:
        row[f"p<{alpha}"] = round(float((d.p < alpha).mean()), 3)
    big = d[d.spikes_used >= 100]
    row["n_>=100_spikes"] = len(big)
    row["p<0.05,>=100_spikes"] = round(float((big.p < 0.05).mean()), 3) if len(big) else np.nan
    row["info>0.5_bits"] = round(float((d["info"] > 0.5).mean()), 3)
    var.append(row)
print(pd.DataFrame(var).to_string(index=False))
print("\nSpatial information of tested ES units, range:", round(es["info"].min(), 3), "to", round(es["info"].max(), 3), "bits/spike")

fig, ax = plt.subplots(figsize=(8, 3.5))
labels = []
for i, ((rat, day, epoch), d) in enumerate(es.groupby(["rat", "day", "epoch"])):
    ax.scatter(np.full(len(d), i) + np.random.default_rng(i).uniform(-0.15, 0.15, len(d)), d["info"], s=10,
               c=np.where(d.sig, "tab:blue", "tab:gray"))
    ax.scatter(np.full(len(d), i) + 0.3, d["null95"], s=6, marker="_", c="k")
    labels.append(f"{rat} {day}\n{epoch.split()[1]}")
ax.set_xticks(range(len(labels)), labels, fontsize=8)
ax.set_ylabel("spatial information (bits/spike)")
ax.set_title("ES units: blue significant, grey not; dashes: 95th percentile of shuffles")
plt.tight_layout()
plt.show()

In [ ]:
# Per-unit results of the two judged sessions, with the Monte Carlo standard error of each p
# (500 shuffles) and the fraction of spikes dropped for lack of a position, real and shifted.
fd9 = es[es.day == "FD9"].copy()
fd9["mc_se"] = np.sqrt(fd9.p * (1 - fd9.p) / nl.N_SHUFFLES)
fd9["spikes_dropped"] = 1 - fd9.spikes_used / fd9.spikes_in_epoch
fd9["shuffle_spikes_dropped"] = 1 - fd9.null_spikes_used / fd9.spikes_in_epoch
cols = ["rat", "tetrode", "cluster", "spikes_in_epoch", "spikes_used", "spikes_dropped", "shuffle_spikes_dropped", "info", "null_mean", "null95", "p", "mc_se", "sig"]
print(fd9.sort_values(["rat", "p"])[cols].round(3).to_string(index=False))
for rat, d in fd9.groupby("rat"):
    print(f"{rat} FD9: units with 0.01 < p < 0.1: {int(((d.p > 0.01) & (d.p < 0.1)).sum())}; "
          f"within 2 Monte Carlo SE of 0.05: {int((np.abs(d.p - 0.05) < 2 * np.maximum(d.mc_se, np.sqrt(0.05 * 0.95 / nl.N_SHUFFLES))).sum())}")

print("\nRepeats of the judged sessions with 2000 shuffles and five other seeds:")
rows = []
for seed in range(5):
    rng_rep = np.random.default_rng([20000301, seed])
    for rat in ["Rat1", "Rat2"]:
        s = by_label[f"{rat} FD9"]
        a, b, name = s["epochs"][s["epochs"].type == "ES"][["start", "stop", "name"]].values[0]
        d = nl.epoch_units(s, a, b, nl.epoch_occupancy(s, name), rng_rep, n_shuffles=2000).dropna(subset=["info"])
        rows.append({"seed": seed, "rat": rat, "tested": len(d), "significant": int((d.p < 0.05).sum()),
                     "fraction": round(float((d.p < 0.05).mean()), 3)})
repeats = pd.DataFrame(rows)
print(repeats.pivot(index="seed", columns="rat", values=["significant", "fraction"]).to_string())


In [ ]:
# Spikes dropped for lack of a position in every task epoch, and the count-matched variant, in which
# each shuffle is compared with the unit's information on a subsample of its own spikes of the same size.
tested["spikes_dropped"] = 1 - tested.spikes_used / tested.spikes_in_epoch
tested["shuffle_spikes_dropped"] = 1 - tested.null_spikes_used / tested.spikes_in_epoch
print(tested.groupby(["type", "rat", "day", "epoch"]).agg(units=("p", "size"), median_spikes_dropped=("spikes_dropped", "median"),
      median_shuffle_spikes_dropped=("shuffle_spikes_dropped", "median"), max_shuffle_spikes_dropped=("shuffle_spikes_dropped", "max"),
      empty_shuffles=("empty_shuffles", "sum"), median_info=("info", "median"), median_null_mean=("null_mean", "median")).round(3).to_string())

rng_cm = np.random.default_rng(20000302)
rows = []
for s in sessions:
    if "xy" not in s:
        continue
    for a, b, typ, name in s["epochs"][["start", "stop", "type", "name"]].itertuples(index=False):
        if typ != "ES":
            continue
        d = nl.epoch_units(s, a, b, nl.epoch_occupancy(s, name), rng_cm, match_counts=True).dropna(subset=["info"])
        main_d = es[es.epoch == f"ES {a:.0f}-{b:.0f}"]
        main_d = main_d[(main_d.rat == s["subject"]) & (main_d.day == s["day"])]
        rows.append({"session": nl.label(s), "epoch": f"ES {a:.0f}-{b:.0f}", "tested": len(d), "significant_main": int(main_d.sig.sum()),
                     "significant_count_matched": int((d.p < 0.05).sum()), "fraction_count_matched": round(float((d.p < 0.05).mean()), 3),
                     "binom_p_vs_0.05": stats.binomtest(int((d.p < 0.05).sum()), len(d), 0.05, alternative="greater").pvalue})
print("\nCount-matched variant, ES epochs:")
print(pd.DataFrame(rows).to_string(index=False))


In [ ]:
# Calibration of the shuffle test at the real drop rates. In every flight ES epoch each unit's spikes
# are replaced by a train without spatial tuning and run through the same test (200 shuffles):
# (a) a Poisson train with the unit's mean count, uniform in the epoch; (b) the same, with each spike
# moved to the time of its nearest position sample, so that, as for real spikes, every spike has a
# sample at its own time and only shifted spikes are dropped for lack of one.
rng_cal = np.random.default_rng(20000303)
rows = []
for s in sessions:
    if "xy" not in s:
        continue
    for a, b, typ, name in s["epochs"][["start", "stop", "type", "name"]].itertuples(index=False):
        if typ != "ES":
            continue
        win = s["t"][(s["t"] >= a) & (s["t"] <= b)]
        for kind in ["poisson", "poisson at sample times"]:
            fake_spikes = []
            for sp in s["units"].spikes:
                t = np.sort(rng_cal.uniform(a, b, rng_cal.poisson(((sp >= a) & (sp <= b)).sum())))
                if kind != "poisson":
                    i = np.clip(np.searchsorted(win, t), 1, len(win) - 1)
                    t = np.where(np.abs(win[i - 1] - t) <= np.abs(win[i] - t), win[i - 1], win[i])
                fake_spikes.append(t)
            fake = dict(s, units=s["units"].assign(spikes=fake_spikes))
            d = nl.epoch_units(fake, a, b, nl.epoch_occupancy(s, name), rng_cal, n_shuffles=200).dropna(subset=["info"])
            k = int((d.p < 0.05).sum())
            ci = stats.binomtest(k, len(d)).proportion_ci() if len(d) else None
            rows.append({"session": nl.label(s), "epoch": f"ES {a:.0f}-{b:.0f}", "null": kind, "tested": len(d), "false_positives": k,
                         "fraction": round(k / len(d), 3) if len(d) else np.nan,
                         "ci95": f"{ci.low:.3f}-{ci.high:.3f}" if ci else "",
                         "median_spikes_dropped": round(float((1 - d.spikes_used / d.spikes_in_epoch).median()), 3),
                         "median_shuffle_spikes_dropped": round(float((1 - d.null_spikes_used / d.spikes_in_epoch).median()), 3),
                         "median_null95": round(float(d.null95.median()), 3)})
cal = pd.DataFrame(rows)
print(cal.to_string(index=False))
pooled = {}
for kind, g in cal.groupby("null"):
    k, n = int(g.false_positives.sum()), int(g.tested.sum())
    pooled[kind] = k / n
    print(f"{kind}: {k} of {n} units significant over all ES epochs ({k / n:.3f}); binomial p against 0.05: "
          f"{stats.binomtest(k, n, 0.05, alternative='greater').pvalue:.3f}")
print("\nThe judged FD9 counts against the false-positive fraction of the same session and the pooled one (one-sided binomial p):")
for rat in ["Rat1", "Rat2"]:
    d = es[(es.rat == rat) & (es.day == "FD9")]
    k, n = int(d.sig.sum()), len(d)
    for kind in pooled:
        f_s = float(cal[(cal.session == f"{rat} FD9") & (cal.null == kind)].fraction.iloc[0])
        print(f"  {rat} FD9 {k} of {n}, null '{kind}': session {f_s:.3f}, p {stats.binomtest(k, n, f_s, alternative='greater').pvalue:.2g}; "
              f"pooled {pooled[kind]:.3f}, p {stats.binomtest(k, n, pooled[kind], alternative='greater').pvalue:.2g}")

In [ ]:
# The judged sessions on the earlier version's files (0.260728.1352), with which the first attempt
# ran: each rat's units and its own old file's position trace, in which a lost position is (0, 255).
# Same test (500 shuffles), the draft's occupancy maps, five seeds.
rows = []
for seed in range(5):
    rng_old = np.random.default_rng([20000305, seed])
    for rat in ["Rat1", "Rat2"]:
        s = by_label[f"{rat} FD9"]
        o = old_files[s["path"]]
        xy = o["xy"].astype(float).copy()
        lost = (xy[:, 0] == 0) & (xy[:, 1] == 255)
        xy[lost] = np.nan
        old_s = dict(s, t=o["t"], xy=xy, valid=~lost,
                     units=pd.DataFrame({"tetrode": [t for t, _ in o["units"]], "cluster": [int(c) for _, c in o["units"]], "spikes": o["spikes"]}))
        a, b, name = s["epochs"][s["epochs"].type == "ES"][["start", "stop", "name"]].values[0]
        d = nl.epoch_units(old_s, a, b, nl.epoch_occupancy(s, name), rng_old).dropna(subset=["info"])
        rows.append({"seed": seed, "rat": rat, "tested": len(d), "significant": int((d.p < 0.05).sum()),
                     "median_spikes_dropped": round(float((1 - d.spikes_used / d.spikes_in_epoch).median()), 3),
                     "median_shuffle_spikes_dropped": round(float((1 - d.null_spikes_used / d.spikes_in_epoch).median()), 3)})
print(f"FD9 ES on {OLD_VERSION}:")
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# Real spike trains decoupled from the position, to see what sets the upper tail of the shuffles.
# (a) In every flight ES epoch, each unit's spikes reversed in time within the epoch (t -> start +
# stop - t) and moved to the nearest position sample: the train keeps its bursts and slow changes in
# rate and loses its relation to the position. (b) In Rat3 FD4, each unit's ES1 spikes run against
# the ES2 trace and its ES2 spikes against the ES1 trace, at the same time from the epoch's start (the
# part beyond the shorter epoch is cut) and moved to the nearest sample. Same test, 200 shuffles.
# Temporal clumping of a train is the effective fraction of 10 s bins it fills, (sum n)^2 /
# sum(n^2) / bins: 1 for an even rate, small when the spikes fall in a few bins.
def to_samples(t, win):
    i = np.clip(np.searchsorted(win, t), 1, len(win) - 1)
    return np.sort(np.where(np.abs(win[i - 1] - t) <= np.abs(win[i] - t), win[i - 1], win[i]))

def spread(sp, a, b, w=10.0):
    n = np.histogram(sp, np.arange(a, b + w, w))[0]
    return n.sum() ** 2 / (n ** 2).sum() / len(n) if n.sum() else np.nan

rng_dec = np.random.default_rng(20000306)
cal_idx = {(r.session, r.epoch): r for r in cal[cal.null == "poisson at sample times"].itertuples()}
rows, dec = [], []
for s in sessions:
    if "xy" not in s:
        continue
    for a, b, typ, name in s["epochs"][["start", "stop", "type", "name"]].itertuples(index=False):
        if typ != "ES":
            continue
        win = s["t"][(s["t"] >= a) & (s["t"] <= b)]
        rev = [to_samples(a + b - sp[(sp >= a) & (sp <= b)], win) for sp in s["units"].spikes]
        d = nl.epoch_units(dict(s, units=s["units"].assign(spikes=rev)), a, b, nl.epoch_occupancy(s, name), rng_dec, n_shuffles=200)
        epoch = f"ES {a:.0f}-{b:.0f}"
        real = tested[(tested.rat == s["subject"]) & (tested.day == s["day"]) & (tested.epoch == epoch)].set_index(["tetrode", "cluster"])
        real = real.assign(spread=[spread(sp[(sp >= a) & (sp <= b)], a, b) for sp in s["units"].set_index(["tetrode", "cluster"]).loc[real.index].spikes])
        real["null95_reversed"] = d.set_index(["tetrode", "cluster"]).null95.reindex(real.index)
        dec.append(real.reset_index().assign(session=nl.label(s), epoch=epoch))
        k, n = int((d.p < 0.05).sum()), int(d.p.notna().sum())
        rows.append({"session": nl.label(s), "epoch": epoch, "tested": len(real), "significant": int(real.sig.sum()),
                     "reversed_tested": n, "reversed_significant": k,
                     "median_null95_real": round(float(real.null95.median()), 3), "median_null95_reversed": round(float(d.null95.median()), 3),
                     "median_null95_poisson_at_samples": round(float(cal_idx[(nl.label(s), epoch)].median_null95), 3),
                     "median_spikes_used": float(real.spikes_used.median()), "median_spread": round(float(real.spread.median()), 3)})
print("(a) Time-reversed real trains, and the median 95th percentile of the shuffles for real, reversed and Poisson trains:")
dec_table = pd.DataFrame(rows)
print(dec_table.to_string(index=False))
print(f"Reversed trains: {dec_table.reversed_significant.sum()} of {dec_table.reversed_tested.sum()} units significant over all ES epochs")

r3 = by_label["Rat3 FD4"]
es3 = r3["epochs"][r3["epochs"].type == "ES"].sort_values("start")
e = es3[["start", "stop"]].values
length = min(e[0, 1] - e[0, 0], e[1, 1] - e[1, 0])
rows = []
for src, dst in [(0, 1), (1, 0)]:
    win = r3["t"][(r3["t"] >= e[dst, 0]) & (r3["t"] <= e[dst, 1])]
    moved = []
    for sp in r3["units"].spikes:
        x = sp[(sp >= e[src, 0]) & (sp <= e[src, 1])] - e[src, 0]
        moved.append(to_samples(e[dst, 0] + x[x < length], win))
    d = nl.epoch_units(dict(r3, units=r3["units"].assign(spikes=moved)), e[dst, 0], e[dst, 1],
                       nl.epoch_occupancy(r3, es3.name.values[dst]), rng_dec, n_shuffles=200).dropna(subset=["info"])
    rows.append({"spikes_of": es3.name.values[src], "trace_of": es3.name.values[dst], "tested": len(d),
                 "significant": int((d.p < 0.05).sum()), "median_null95": round(float(d.null95.median()), 3)})
print("\n(b) Rat3 FD4, each epoch's spikes against the other epoch's trace:")
print(pd.DataFrame(rows).to_string(index=False))

D = pd.concat(dec, ignore_index=True).dropna(subset=["null95_reversed"])
X = np.column_stack([np.ones(len(D)), np.log(D.spikes_used), np.log(D.spread)])
coef, *_ = np.linalg.lstsq(X, np.log(D.null95), rcond=None)
r2 = 1 - np.var(np.log(D.null95) - X @ coef) / np.var(np.log(D.null95))
print(f"\nOver {len(D)} tested flight ES units: Spearman rho of the real null's 95th percentile with the reversed train's "
      f"{stats.spearmanr(D.null95, D.null95_reversed).statistic:.3f}, with spikes used {stats.spearmanr(D.null95, D.spikes_used).statistic:.3f}, "
      f"with spread {stats.spearmanr(D.null95, D.spread).statistic:.3f}")
print(f"log null95 = {coef[0]:.2f} + {coef[1]:.3f} log(spikes used) + {coef[2]:.3f} log(spread), R^2 {r2:.2f}")
print("\nUnits named in the round-2 verification (spread and 95th percentile in each epoch):")
print(D[D.set_index(["session", "tetrode", "cluster"]).index.isin([("Rat3 FD4", "TT4", 1), ("Rat3 FD4", "TT5", 1), ("Rat3 FD4", "TT3", 2), ("Rat2 FD9", "TT13", 11)])]
      [["session", "epoch", "tetrode", "cluster", "spikes_used", "spread", "info", "null95", "null95_reversed", "p"]].round(3).to_string(index=False))

In [ ]:
print("Attribution contrast from the data check, ES epochs:")
print(contrast.to_string(index=False))

**Result: not reproduced.** On flight day 9, 10 of 25 Rat1 units (0.40) have significant spatial
information on the Escher staircase (binomial p = 1.6e-7 against 0.05), and 2 of 28 Rat2 units
(0.071, p = 0.41). By the criterion, Rat1 is above chance but below 0.5 and Rat2 is not above
chance, so neither the rule for reproduced nor the rule for partially reproduced is met. Repeats
with 2000 shuffles and five other seeds give 10 or 11 of 25 for Rat1 (0.40 to 0.44) and 2 of 28 for
Rat2; the run of 200 shuffles in the data check gives 10 of 25 and 2 of 28; 5 Rat1 units have p
between 0.01 and 0.1, 3 within two Monte Carlo standard errors of 0.05. On the files of 0.260728.1352 the same test gives
11 or 12 of 25 for Rat1 and 2 or 3 of 28 for Rat2 over five seeds; the first attempt recorded 12 of
25 (11 to 13 over seven runs). Every real spike has a sample at its own time in
both versions; the draft's trace adds samples at the other rat's spike times in each window, so a
shifted spike finds a sample within 0.25 s more often and the shuffle distributions change. Shifted spikes are now dropped about as often in Rat2's FD9 ES epoch (median 0.058)
as in Rat1's (0.060), against 0.306 to 0.307 and 0.074 to 0.075 on the earlier version's files, so the drops no longer differ
between the rats; the count-matched variant gives 12 of 25 (0.48) for Rat1 and 2 of 28 for Rat2,
still below 0.5. The test was calibrated at the real drop rates by replacing each unit's spikes in
every flight ES epoch with a Poisson train of the same mean count: 9 of 122 units are significant
over all ES epochs (0.074, binomial p = 0.16 against 0.05), and 12 of 125 (0.096, p = 0.023) when
each spike is moved to its nearest position sample, so that shifted spikes are dropped more often
than the observed ones, as in the real data. Per session the false-positive fraction is 0 to 0.21;
in the judged sessions it is 0.160 and 0.154 for Rat1 and 0.037 and 0.214 for Rat2. The criterion's
chance level of 0.05 is thus below the false-positive fraction of some sessions. Against each
session's own fractions, Rat1's 10 of 25 is still above chance (p = 0.0035 and 0.0026) and Rat2's
2 of 28 is not (p = 0.28 and 0.99), so the outcome is the same.

Poisson trains are not a good model of the real trains, whose spikes come in bursts and in
periods of high and low rate. When each unit's real train is reversed in time within its epoch, so
that it keeps this structure and loses its relation to the position, 2 of 122 units are significant
over all flight ES epochs (1 of 25 in Rat1's FD9 session, 0 of 27 in Rat2's), and Rat3's ES1 trains
against the ES2 trace and the reverse give 0 of 16 and 0 of 19. So with real temporal structure the
test is conservative, not lenient. The 95th percentile of a unit's shuffles is set by its train: it
is nearly the same for the reversed train (Spearman rho 0.995 over 121 units) and 1.7 to 2.3 times that
of a Poisson train of the same count in each session (medians 2.141 to 3.572 against 1.070 to 1.538). It
is higher the fewer 10 s periods hold a unit's spikes (rho -0.874 with the spread, the effective
fraction of 10 s bins filled) and the fewer spikes it has (rho -0.601); the two explain 80% of its
variance on a log scale. Rat2's FD9 units are the most clumped (median spread 0.145, against 0.190
for Rat1 FD9 and 0.222 to 0.358 elsewhere) and have the highest median 95th percentile (3.572); TT13
cluster 11 has a spread of 0.064 and a 95th percentile of 7.928. A unit whose spikes fall in a few
periods needs much more information to pass, so the test has less power for such units, and this
power was not measured. A low fraction of significant units, Rat2's especially, is therefore weak
evidence against a spatial code. The criterion's rule still decides the outcome. Rat2's units are not selective with the trace in any window, and Rat1's are
selective in Rat2's FD4 ES epoch (10 of 15), so Rat2's result may come from a trace that is not
Rat2's position; the files do not show whose position it is. On flight day 4, 11 of 20 Rat1 units,
1 of 16 Rat2 units, and 11 of 16 and 1 of 19 Rat3 units in its ES1 and ES2 epochs are significant.
In the MC epochs, which the claim does not cover, 0 to 2 units per session are significant.

## three-orthogonal-surfaces

> The spatial code covers each of the three orthogonal surfaces of the Escher staircase. (Abstract)

Blocked. The dandiset holds one two-dimensional pixel position from one camera; the NASA record
of the study (OSD-968) says position was recorded with two synchronised cameras (printed below).
The draft's position reference frame says the camera view and the pixel size on the apparatus are
not recorded in the source files (data check), nothing in the files or the dandiset metadata
describes the track layout in the image, and the conversion code describes the ES track only as a
three-dimensional track with 90 degree yaw and pitch turns and the position only as video tracker
pixel coordinates (lines printed below), so a pixel cannot be assigned to one of the three
surfaces.

In [ ]:
# The NASA record of the study (OSD-968): what it says of the cameras, its description of the files,
# and its file listing. Then every line of the conversion code that describes the track or the position.
osd_text = json.dumps(requests.get("https://osdr.nasa.gov/osdr/data/osd/meta/968", timeout=60).json(), ensure_ascii=False)
for word in ["camera", "Raw (Recording) Data Files", "EEG", "theta", "ripple", "LFP"]:
    hits = [m.group(0) for m in re.finditer(r"[^.\"]{0,150}" + re.escape(word) + r"[^\"]{0,450}", osd_text)]
    print(f"OSD-968 metadata, '{word}': {len(hits)} passage(s)")
    for h in hits[:1]:
        print("   ", h[:600])
files = requests.get("https://osdr.nasa.gov/osdr/data/osd/files/968", timeout=60).json()["studies"]["OSD-968"]["study_files"]
print(f"\nOSD-968 files ({len(files)}):")
for f in files:
    print(f"  {f['file_name']}  {f['file_size'] / 1e6:.1f} MB")

repo, commit = "catalystneuro/mcnaughton2000-to-nwb", "734688253c17a32e26d84c57f41fbe48ffbcd009"
tree = requests.get(f"https://api.github.com/repos/{repo}/git/trees/{commit}?recursive=1", timeout=60).json()["tree"]
paths = [t["path"] for t in tree if t["type"] == "blob" and t["path"].endswith((".py", ".md"))]
print(f"\nConversion code at {commit[:7]}, every line on the track, camera, surfaces, coordinates or waveforms:")
for path in paths:
    text = requests.get(f"https://raw.githubusercontent.com/{repo}/{commit}/{path}", timeout=60).text
    for n, line in enumerate(text.splitlines(), 1):
        if re.search(r"camera|surface|staircase|layout|geometry|reference_frame|coordinate|waveform", line, re.I):
            print(f"  {path}:{n}: {line.strip()[:160]}")


## si-comparable-to-preflight

> In flight, spatial information content of place cells is comparable to preflight levels.
> (Dandiset description)

Criterion: two-sided Mann-Whitney U test at alpha 0.05 of the spatial information of the
dandiset's rate maps, FD9 ES against preflight ES, Rats 1 and 2 pooled, every distinct map with a
positive mean rate. Reproduced when the test finds no difference.

In [ ]:
m = maps[(~maps.duplicate) & (maps.mean_rate > 0)].copy()
print("Maps dropped: duplicates", int(maps.duplicate.sum()), "; zero mean rate", int(((~maps.duplicate) & (maps.mean_rate <= 0)).sum()),
      "; kept", len(m), "of", len(maps))
print("Spatial information of rate maps, range:", round(m["info"].min(), 3), "to", round(m["info"].max(), 3), "bits/spike")
print(m.groupby(["type", "rat", "day"])["info"].describe()[["count", "50%", "25%", "75%", "min", "max"]].round(3).to_string())

def compare(a, b, alternative="two-sided"):
    if len(a) == 0 or len(b) == 0:
        return {"n_a": len(a), "n_b": len(b)}
    u = stats.mannwhitneyu(a, b, alternative=alternative)
    return {"n_a": len(a), "n_b": len(b), "median_a": round(float(np.median(a)), 3), "median_b": round(float(np.median(b)), 3),
            "diff": round(float(np.median(a) - np.median(b)), 3), "ratio": round(float(np.median(a) / np.median(b)), 2),
            "U": float(u.statistic), "p": float(u.pvalue), "rank_biserial": round(1 - 2 * u.statistic / (len(a) * len(b)), 3)}

sel = lambda d, typ, days, rats: d[(d.type == typ) & d.day.isin(days) & d.rat.isin(rats)]["info"].values
main = compare(sel(m, "ES", ["FD9"], ["Rat1", "Rat2"]), sel(m, "ES", ["preflight"], ["Rat1", "Rat2"]))
print("\nMain test, ES FD9 (a) against ES preflight (b), Rats 1 and 2:", main)

In [ ]:
rows = []
variants = {
    "per rat Rat1": (m, "ES", ["FD9"], ["Rat1"], ["preflight"], ["Rat1"]),
    "per rat Rat2": (m, "ES", ["FD9"], ["Rat2"], ["preflight"], ["Rat2"]),
    "CEL names only": (m[m.cel_name], "ES", ["FD9"], ["Rat1", "Rat2"], ["preflight"], ["Rat1", "Rat2"]),
    "peak >= 1 Hz": (m[m.peak_rate >= 1], "ES", ["FD9"], ["Rat1", "Rat2"], ["preflight"], ["Rat1", "Rat2"]),
    "MC maps": (m, "MC", ["FD9"], ["Rat1", "Rat2"], ["preflight"], ["Rat1", "Rat2"]),
    "all flight ES (FD4+FD9, 3 rats)": (m, "ES", ["FD4", "FD9"], ["Rat1", "Rat2", "Rat3"], ["preflight"], ["Rat1", "Rat2", "Rat3"]),
}
for name, (d, typ, da, ra, db, rb) in variants.items():
    a = d[(d.type == typ) & d.day.isin(da) & d.rat.isin(ra)]["info"].values
    b = d[(d.type == typ) & d.day.isin(db) & d.rat.isin(rb)]["info"].values
    rows.append({"variant": name} | compare(a, b))
print("Variants (a = flight, b = preflight):")
print(pd.DataFrame(rows).to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 3.5))
groups = [(f"{rat}\n{day}", m[(m.type == "ES") & (m.rat == rat) & (m.day == day)]["info"].values)
          for rat in ["Rat1", "Rat2", "Rat3"] for day in ["preflight", "FD4", "FD9"]]
groups = [(g, v) for g, v in groups if len(v)]
ax.boxplot([v for _, v in groups], showfliers=True)
ax.set_xticks(range(1, len(groups) + 1), [g for g, _ in groups], fontsize=8)
ax.set_ylabel("spatial information of ES rate maps (bits/spike)")
plt.tight_layout()
plt.show()

In [ ]:
# Sampling bias of spatial information: epoch durations and spike counts per ES epoch (all files,
# from the units), the rate maps' total occupancy and rate x occupancy (proportional to the spike
# count where the frame rate is the same; see the data check), and two variants that limit the bias.
rows = []
for s in sessions:
    for a, b, typ in s["epochs"][["start", "stop", "type"]].itertuples(index=False):
        if typ != "ES":
            continue
        n = np.array([int(((sp >= a) & (sp <= b)).sum()) for sp in s["units"].spikes])
        rows.append({"session": nl.label(s), "epoch": f"ES {a:.0f}-{b:.0f}", "duration_s": b - a, "units_with_spikes": int((n > 0).sum()),
                     "units_>=50_spikes": int((n >= 50).sum()), "median_spikes": float(np.median(n[n > 0])),
                     "median_rate_hz": round(float(np.median(n[n > 0]) / (b - a)), 2)})
print("Unit spike counts in each ES epoch (all units with a spike in the epoch):")
print(pd.DataFrame(rows).to_string(index=False))

mes = m[m.type == "ES"].copy()
print("\nES rate maps per session:")
print(mes.groupby(["rat", "day"]).agg(maps=("info", "size"), median_occupancy_total=("occupancy_total", "median"),
      median_rate_x_occupancy=("rate_x_occupancy", "median"), median_info=("info", "median"),
      median_info_occ5=("info_occ5", "median")).round(3).to_string())
x = np.log(mes.rate_x_occupancy)
print(f"\nSpearman rho of info with log(rate x occupancy), all ES maps: {stats.spearmanr(x, mes['info']).statistic:.3f} (n {len(mes)})")
print("Within each session:", {f"{r} {d}": round(float(stats.spearmanr(np.log(g.rate_x_occupancy), g['info']).statistic), 3) for (r, d), g in mes.groupby(["rat", "day"])})
fit = np.polyfit(x, mes["info"], 1)
print(f"Linear fit over all ES maps: info = {fit[1]:.3f} + {fit[0]:.3f} x log(rate x occupancy)")
m["info_adj"] = m["info"] - np.polyval(fit, np.log(m.rate_x_occupancy))

rows = []
rows.append({"variant": "info over bins with occupancy >= 5"} | compare(sel(m.assign(info=m.info_occ5), "ES", ["FD9"], ["Rat1", "Rat2"]), sel(m.assign(info=m.info_occ5), "ES", ["preflight"], ["Rat1", "Rat2"])))
rows.append({"variant": "count-adjusted info (residual of the fit)"} | compare(sel(m.assign(info=m.info_adj), "ES", ["FD9"], ["Rat1", "Rat2"]), sel(m.assign(info=m.info_adj), "ES", ["preflight"], ["Rat1", "Rat2"])))
print("\nBias variants, FD9 ES (a) against preflight ES (b), Rats 1 and 2, two-sided:")
print(pd.DataFrame(rows).to_string(index=False))


In [ ]:
# Does the rate maps' spatial information separate the units that pass the spike-based shuffle test
# from those that do not? Each tested flight ES unit joined to its rate map (same rat, day, epoch
# name, tetrode and cluster). Then a bootstrap 95% interval for the main test's median difference.
name_of = {(s["subject"], s["day"], f"{t} {a:.0f}-{b:.0f}"): n for s in sessions
           for a, b, t, n in s["epochs"][["start", "stop", "type", "name"]].itertuples(index=False)}
j = tested[tested.type == "ES"].copy()
j["epoch_name"] = [name_of[k] for k in zip(j.rat, j.day, j.epoch)]
j = j.merge(m[["rat", "day", "epoch_name", "tetrode", "cluster", "info"]].rename(columns={"info": "map_info"}),
            on=["rat", "day", "epoch_name", "tetrode", "cluster"], how="left")
print(f"Tested flight ES units: {len(j)}; with a rate map: {int(j.map_info.notna().sum())}")
rows = []
for key, g in list(j.groupby(["rat", "day", "epoch_name"])) + [(("all", "", ""), j)]:
    g = g.dropna(subset=["map_info"])
    a, b = g[g.sig].map_info.values, g[~g.sig].map_info.values
    rows.append({"session": " ".join(key).strip(), "significant": len(a), "not_significant": len(b),
                 "median_map_info_sig": round(float(np.median(a)), 3) if len(a) else np.nan,
                 "median_map_info_not_sig": round(float(np.median(b)), 3) if len(b) else np.nan,
                 "mann_whitney_p": stats.mannwhitneyu(a, b).pvalue if len(a) and len(b) else np.nan})
print(pd.DataFrame(rows).to_string(index=False))
jj = j.dropna(subset=["map_info"])
print(f"Spearman rho of rate-map information with spike-based information: {stats.spearmanr(jj.map_info, jj['info']).statistic:.3f}; "
      f"with spike-based information minus the shuffle mean: {stats.spearmanr(jj.map_info, jj['info'] - jj.null_mean).statistic:.3f} (n {len(jj)})")

rng_bs = np.random.default_rng(20000304)
for rats in (["Rat1", "Rat2"], ["Rat1"]):
    a, b = sel(m, "ES", ["FD9"], rats), sel(m, "ES", ["preflight"], rats)
    diffs = np.array([np.median(rng_bs.choice(a, len(a))) - np.median(rng_bs.choice(b, len(b))) for _ in range(10000)])
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    print(f"FD9 minus preflight median, {'+'.join(rats)}: {np.median(a) - np.median(b):.3f} bits/spike, bootstrap 95% interval "
          f"{lo:.3f} to {hi:.3f} ({lo / np.median(b):+.0%} to {hi / np.median(b):+.0%} of the preflight median)")

**Result: reproduced.** Flight day 9 ES maps (64, Rats 1 and 2) have a median of 1.385 bits/spike
against 1.448 for preflight ES maps (53); Mann-Whitney p = 0.45, a difference of -0.064 bits/spike
(ratio 0.96, rank-biserial 0.081). The main test's maps and numbers are the same as in
0.260728.1352. Every
variant gives p of 0.12 or more: per rat (0.25 and 0.88), maps with archived names containing CEL
(0.43), maps with a peak of 1 Hz or more (0.89), MC maps (0.92), all flight maps (0.12), bins with 5
or more occupancy counts (0.44), and spatial information adjusted for rate x occupancy (0.37), which
is proportional to the spike count in the flight maps (about 22 per spike, data check) and is
assumed to be so in the preflight maps, where it cannot be checked. Spatial information falls as the
spike count rises, as sampling bias predicts (Spearman rho -0.226 over all ES maps; -0.118 to -0.792
within seven files, 0.042 in Rat3's preflight). Rat1's compared sessions have similar spike counts
and Rat2's do not: units in Rat1's and Rat2's preflight ES epochs have medians of 359 and 352 spikes
in 465 and 560 s, and on FD9 437 and 205 in 1611 and 2664 s; the maps' median rate x occupancy is
7862 and 7628 preflight against 9105 and 5226 on FD9. Rat2's FD9 maps thus rest on fewer spikes than
its preflight maps, which by sampling bias raises their spatial information. The rate maps are
smoothed with an unrecorded kernel, so no shuffle-corrected value can be computed for them. The
dandiset's description says the preflight recordings were made on a flat rectangular track, so this
may be a comparison between different apparatus, and Rat2's preflight file shares no tetrode with
its flight files (see the data check). Rat2's FD9 maps (median 1.343 bits/spike) rest on a trace
that could not be attributed to Rat2, while only 2 of its 28 units are significant against shuffles,
so the rate maps' spatial information does not show that Rat2's cells were selective. Over all 124
tested flight ES units, the rate maps of units significant against their shuffles have a higher
median than the others (1.855 against 1.076 bits/spike, Mann-Whitney p = 9.0e-7; Spearman rho 0.791
with spike-based information), so the measure does track selectivity, but Rat2's 26 FD9 units that
are not significant have a median of 1.283, close to the preflight median of 1.448. The bootstrap
95% interval of the FD9 minus preflight median difference is -0.425 to 0.312 bits/spike (-29% to
+22% of the preflight median); for Rat1 alone, whose trace is attributed, it is -0.750 to 0.543 (p =
0.25). The criterion's test has no equivalence margin, and a drop of up to 29% of the preflight median
(40% for Rat1 alone) is not excluded, so the result shows that no difference was detected.

## fd4-abnormal-selectivity

> On the first in-flight exposures (flight day 4) spatial selectivity was abnormal. (Dandiset
> description)

Criterion: abnormal is read as lower spatial information. FD4 ES rate maps (Rats 1 to 3) against
preflight ES rate maps (Rats 1 to 3); reproduced when the FD4 median is lower and a one-sided
Mann-Whitney U test gives p < 0.05.

In [ ]:
fd4 = sel(m, "ES", ["FD4"], ["Rat1", "Rat2", "Rat3"])
pre = sel(m, "ES", ["preflight"], ["Rat1", "Rat2", "Rat3"])
main4 = compare(fd4, pre, alternative="less")
print("Main test, ES FD4 (a) below ES preflight (b), Rats 1 to 3, one-sided:", main4)
rows = []
for rat in ["Rat1", "Rat2", "Rat3"]:
    rows.append({"comparison": f"{rat} FD4 < preflight"} | compare(sel(m, "ES", ["FD4"], [rat]), sel(m, "ES", ["preflight"], [rat]), "less"))
for rat in ["Rat1", "Rat2"]:
    rows.append({"comparison": f"{rat} FD4 < FD9"} | compare(sel(m, "ES", ["FD4"], [rat]), sel(m, "ES", ["FD9"], [rat]), "less"))
rows.append({"comparison": "Rats 1-2 FD4 < FD9"} | compare(sel(m, "ES", ["FD4"], ["Rat1", "Rat2"]), sel(m, "ES", ["FD9"], ["Rat1", "Rat2"]), "less"))
rows.append({"comparison": "CEL names, FD4 < preflight"} | compare(sel(m[m.cel_name], "ES", ["FD4"], ["Rat1", "Rat2", "Rat3"]), sel(m[m.cel_name], "ES", ["preflight"], ["Rat1", "Rat2", "Rat3"]), "less"))
rows.append({"comparison": "peak >= 1 Hz, FD4 < preflight"} | compare(sel(m[m.peak_rate >= 1], "ES", ["FD4"], ["Rat1", "Rat2", "Rat3"]), sel(m[m.peak_rate >= 1], "ES", ["preflight"], ["Rat1", "Rat2", "Rat3"]), "less"))
rows.append({"comparison": "MC maps, FD4 < preflight"} | compare(sel(m, "MC", ["FD4"], ["Rat1", "Rat2", "Rat3"]), sel(m, "MC", ["preflight"], ["Rat1", "Rat2", "Rat3"]), "less"))
print(pd.DataFrame(rows).to_string(index=False))

print("\nSpike-based fraction of significant ES units by day (from flight-spatial-code):")
print(es.groupby(["day", "rat"]).agg(tested=("sig", "size"), significant=("sig", "sum"), fraction=("sig", "mean"),
      median_info=("info", "median")).round(3).to_string())
k4, n4 = int(es[es.day == "FD4"].sig.sum()), int((es.day == "FD4").sum())
k9, n9 = int(es[es.day == "FD9"].sig.sum()), int((es.day == "FD9").sum())
print(f"FD4 {k4}/{n4} against FD9 {k9}/{n9}; Fisher exact one-sided (FD4 lower) p = "
      f"{stats.fisher_exact([[k4, n4 - k4], [k9, n9 - k9]], alternative='less')[1]:.3g}")
print("\nVariants for the spatial-information bias (see si-comparable-to-preflight):")
rows = []
rows.append({"comparison": "info over bins with occupancy >= 5, FD4 < preflight"} | compare(sel(m.assign(info=m.info_occ5), "ES", ["FD4"], ["Rat1", "Rat2", "Rat3"]), sel(m.assign(info=m.info_occ5), "ES", ["preflight"], ["Rat1", "Rat2", "Rat3"]), "less"))
rows.append({"comparison": "count-adjusted info, FD4 < preflight"} | compare(sel(m.assign(info=m.info_adj), "ES", ["FD4"], ["Rat1", "Rat2", "Rat3"]), sel(m.assign(info=m.info_adj), "ES", ["preflight"], ["Rat1", "Rat2", "Rat3"]), "less"))
corr_es = es.assign(info=es["info"] - es.null_mean)
rows.append({"comparison": "spike-based info minus shuffle mean, Rats 1-2 FD4 < FD9"} | compare(corr_es[(corr_es.day == "FD4") & corr_es.rat.isin(["Rat1", "Rat2"])]["info"].values, corr_es[corr_es.day == "FD9"]["info"].values, "less"))
rows.append({"comparison": "spike-based info minus shuffle mean, all FD4 < FD9"} | compare(corr_es[corr_es.day == "FD4"]["info"].values, corr_es[corr_es.day == "FD9"]["info"].values, "less"))
print(pd.DataFrame(rows).to_string(index=False))


In [ ]:
# The same test on the rate maps of the earlier version (0.260728.1352), each distinct map once, as
# the earlier attempt counted them: the draft leaves out every copy in the six groups of identical maps.
rows = []
for s in sessions:
    o = old_files[s["path"]]
    in_draft = {np.where(np.isnan(r), -999, r).astype(np.float32).tobytes() for r in s["rate_maps"]}
    seen = set()
    for r_map, occ, typ, tt, f in zip(o["maps"], o["occ"], o["map_types"], o["map_tetrodes"], o["map_files"]):
        if r_map.tobytes() in seen:
            continue
        seen.add(r_map.tobytes())
        info, mean, _ = nl.rma_info(np.where(r_map == -999, np.nan, r_map), occ)
        rows.append({"rat": s["subject"], "day": s["day"], "type": typ, "map": f"{tt}/{f}", "info": info, "mean_rate": mean,
                     "in_draft": r_map.tobytes() in in_draft})
old_maps = pd.DataFrame(rows)
old_maps = old_maps[old_maps.mean_rate > 0]
print(f"Distinct maps with a positive mean rate: {len(old_maps)} in {OLD_VERSION}, {len(m)} in the draft")
print("Distinct maps of the earlier version not in the draft:")
print(old_maps[~old_maps.in_draft].round(3).to_string(index=False))
print("ES maps per rat and day, earlier version and draft:")
print(pd.concat({OLD_VERSION: old_maps[old_maps.type == "ES"].groupby(["rat", "day"]).size(),
                 "draft": m[m.type == "ES"].groupby(["rat", "day"]).size()}, axis=1).to_string())
rats3 = ["Rat1", "Rat2", "Rat3"]
rows = [{"maps": f"{OLD_VERSION}, all distinct"} | compare(sel(old_maps, "ES", ["FD4"], rats3), sel(old_maps, "ES", ["preflight"], rats3), "less"),
        {"maps": f"{OLD_VERSION}, those in the draft"} | compare(sel(old_maps[old_maps.in_draft], "ES", ["FD4"], rats3), sel(old_maps[old_maps.in_draft], "ES", ["preflight"], rats3), "less"),
        {"maps": "draft"} | main4]
print("\nMain test (FD4 ES below preflight ES, Rats 1 to 3, one-sided) on each set of maps:")
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# The main test without Rat2, whose position trace could not be attributed (data check).
rows = [{"comparison": "Rats 1 and 3, FD4 < preflight"} | compare(sel(m, "ES", ["FD4"], ["Rat1", "Rat3"]), sel(m, "ES", ["preflight"], ["Rat1", "Rat3"]), "less"),
        {"comparison": "Rats 1 and 3, FD4 < preflight, old version, all distinct maps"} | compare(sel(old_maps, "ES", ["FD4"], ["Rat1", "Rat3"]), sel(old_maps, "ES", ["preflight"], ["Rat1", "Rat3"]), "less"),
        {"comparison": "Rat1, FD9 against preflight (two-sided)"} | compare(sel(m, "ES", ["FD9"], ["Rat1"]), sel(m, "ES", ["preflight"], ["Rat1"]))]
print(pd.DataFrame(rows).to_string(index=False))

**Result: reproduced.** Flight day 4 ES maps (104, Rats 1 to 3) have a median of 1.177 bits/spike
against 1.441 for preflight (98); the one-sided Mann-Whitney p is 0.024, below the criterion's 0.05
(ratio 0.82, rank-biserial 0.162). The outcome differs from the first attempt's (p = 0.055) because
of three maps: the draft leaves out every copy in the groups of identical maps, and three Rat2 FD4 ES
maps the first attempt counted once each (7.583 bits/spike each, mean rate below 0.001 Hz) are no longer
in the dandiset. On the earlier version's maps the same test gives p = 0.055 with all 107 distinct
maps and p = 0.024 with the 104 that are in the draft, so with Rat2 included the result rests on whether those three
maps are counted. Without Rat2, whose trace could not be attributed, Rats 1 and 3 give 1.242 against
1.510 (p = 0.035), the same on the earlier version, as the three maps are Rat2's. The FD4 median is lower in each rat (ratios 0.65 to 0.88), significantly only for
Rat1 (p = 0.032; Rat2 p = 0.12, Rat3 p = 0.18). The test also passes with maps of peak rate 1 Hz or
more (p = 0.0095), over bins with 5 or more occupancy counts (p = 0.023) and with spatial information
adjusted for rate x occupancy (p = 0.0011), and not with maps whose archived names contain CEL (p =
0.073) or with MC maps (p = 0.060). Rat1's and Rat2's FD4 maps have less rate x occupancy, and so
fewer spikes, than their preflight maps (medians 1891 and 1819 against 7862 and 7628), and their
spatial information is lower even so: sampling bias works against the claim here. For Rats 1 and 2,
FD4 is lower than FD9 without significance on the rate maps (1.081 against 1.385, p = 0.095) and on
spike-based information minus the shuffle mean (0.025 against 0.211, p = 0.13). With Rat3's FD4
units added, that measure is higher on FD4 than on FD9 (0.232 against 0.211, p = 0.31). The spike-based fraction of significant units is not lower on FD4 (24 of
71 against 12 of 53 on FD9, p = 0.94). The claim's "abnormal" may refer to something other than lower
spatial information, which the description does not say, and the preflight maps may come from
another apparatus (see the data check).

## fig3-stability-across-sessions

> Place fields are stable across sessions. (Figure 3 title; which sessions the figure compares is
> not known)

Criterion: Rat3 FD4, its two ES epochs. Same tetrode and cluster id in both, at least 50 spikes at
a known position in each. Smoothed 32 x 32 rate maps; median same-unit correlation against 1000
random re-pairings. Reproduced when above the 95th percentile of the re-paired medians and above
0.5. A unit is the same sorted cluster in both epochs, as the units' description states; that it is
the same neuron is assumed (see the data check).

In [ ]:
rng_c4 = np.random.default_rng(72910)
s3 = by_label["Rat3 FD4"]
es_epochs = s3["epochs"][s3["epochs"].type == "ES"].sort_values("start")
lookup3 = nl.PositionLookup(s3)
maps3, occs = {}, []
for k, (a, b, name) in enumerate(zip(es_epochs.start, es_epochs.stop, es_epochs.name)):
    occ = nl.epoch_occupancy(s3, name)
    occs.append(nl.merge_bins(nl.clean_occupancy(occ)))
    print(f"ES epoch {k + 1}: {name} {a:.0f}-{b:.0f} s, occupancy map match {nl.occupancy_match(s3, a, b)}")
    for _, u in s3["units"].iterrows():
        sp = u.spikes[(u.spikes >= a) & (u.spikes <= b)]
        found, r, c = lookup3(sp)
        if found.sum() >= nl.MIN_SPIKES:
            maps3[(u.tetrode, u.cluster, k)] = nl.smoothed_rate_map(r, c, occ)[0]
both = sorted({(t, c) for t, c, k in maps3 if (t, c, 0) in maps3 and (t, c, 1) in maps3})
only = sorted({(t, c) for t, c, k in maps3} - set(both))
print(f"Units with >= {nl.MIN_SPIKES} spikes in both epochs: {len(both)}; in one only: {len(only)}; units in file: {len(s3['units'])}")
shared = (occs[0] >= nl.MIN_OCC) & (occs[1] >= nl.MIN_OCC)
print("Bins with occupancy >= 5 in both epochs:", int(shared.sum()))

def corr(m1, m2):
    ok = shared & np.isfinite(m1) & np.isfinite(m2)
    return np.corrcoef(m1[ok], m2[ok])[0, 1]

A = [maps3[(t, c, 0)] for t, c in both]
B = [maps3[(t, c, 1)] for t, c in both]
C = np.array([[corr(a, b) for b in B] for a in A])
same = np.diag(C)
for (t, c), r in zip(both, same):
    print(f"  {t} cluster {c}: r = {r:.3f}")
null = np.array([np.median(C[np.arange(len(both)), rng_c4.permutation(len(both))]) for _ in range(1000)])
print(f"\nMedian same-unit r = {np.median(same):.3f} (range {same.min():.3f} to {same.max():.3f}); "
      f"re-paired medians: mean {null.mean():.3f}, 95th percentile {np.percentile(null, 95):.3f}; "
      f"p = {(np.sum(null >= np.median(same)) + 1) / 1001:.4f}")
off = C[~np.eye(len(both), dtype=bool)]
print(f"Different-unit pairs: median r {np.median(off):.3f}, range {off.min():.3f} to {off.max():.3f}")
print("Same-unit r above 0.5:", int((same > 0.5).sum()), "of", len(same), "; above 0.3:", int((same > 0.3).sum()))

print("\nVariants of the smoothing and the spike threshold (same re-pairing test, 1000 re-pairings each):")
rows = []
for sigma in [0.5, 1.0, 2.0]:
    for min_spikes in [50, 100]:
        mm = {}
        for k, (a, b, name) in enumerate(zip(es_epochs.start, es_epochs.stop, es_epochs.name)):
            occ = nl.epoch_occupancy(s3, name)
            for _, u in s3["units"].iterrows():
                sp = u.spikes[(u.spikes >= a) & (u.spikes <= b)]
                found, r, c = lookup3(sp)
                if found.sum() >= min_spikes:
                    mm[(u.tetrode, u.cluster, k)] = nl.smoothed_rate_map(r, c, occ, sigma)[0]
        bb = sorted({(t, c) for t, c, k in mm if (t, c, 0) in mm and (t, c, 1) in mm})
        CC = np.array([[corr(mm[(t1, c1, 0)], mm[(t2, c2, 1)]) for t2, c2 in bb] for t1, c1 in bb])
        nn = np.array([np.median(CC[np.arange(len(bb)), rng_c4.permutation(len(bb))]) for _ in range(1000)])
        rows.append({"sigma_bins": sigma, "min_spikes": min_spikes, "units": len(bb), "median_r": round(float(np.median(np.diag(CC))), 3),
                     "null_95th": round(float(np.percentile(nn, 95)), 3), "p": round((np.sum(nn >= np.median(np.diag(CC))) + 1) / 1001, 4)})
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# Per-session results, and the variant on the dandiset's own rate maps: the ES1 and ES2 maps that
# the rate-map table links to the same unit.
per_epoch = tested[(tested.rat == "Rat3") & (tested.day == "FD4") & (tested.type == "ES")]
print(per_epoch.groupby("epoch").agg(tested=("sig", "size"), significant=("sig", "sum"), median_info=("info", "median")).round(3).to_string())

mp = s3["maps"]
rows = []
for unit, g in mp[mp.type == "ES"].groupby("unit"):
    if {"ES1", "ES2"} <= set(g.epoch_name):
        i, j = g.index[g.epoch_name == "ES1"][0], g.index[g.epoch_name == "ES2"][0]
        r1, r2 = s3["rate_maps"][i], s3["rate_maps"][j]
        ok = np.isfinite(r1) & np.isfinite(r2) & (s3["occupancy_maps"][i] > 0) & (s3["occupancy_maps"][j] > 0)
        ok[nl.SENTINEL_BIN] = False
        rows.append({"tetrode": mp.tetrode[i], "cluster": int(mp.cluster[i]), "in_main_test": (mp.tetrode[i], mp.cluster[i]) in both,
                     "r": np.corrcoef(r1[ok], r2[ok])[0, 1]})
rv = pd.DataFrame(rows)
print("\nDandiset rate maps, ES1 against ES2 map of the same unit (linked by the table's unit column):")
print(rv.round(3).to_string(index=False))
print(f"median r {rv.r.median():.3f} over {len(rv)} units with both maps; over the {int(rv.in_main_test.sum())} units of the main test "
      f"{rv[rv.in_main_test].r.median():.3f}; units of the main test without both maps: {len(both) - int(rv.in_main_test.sum())}")

In [ ]:
# Per unit in Rat3's two FD4 ES epochs: spikes, spike-based information and shuffle drops in each,
# and the same-unit correlation; then the stability of mean rates as a check on cluster identity.
ep_names = list(es_epochs.apply(lambda e: f"ES {e.start:.0f}-{e.stop:.0f}", axis=1))
u3 = units[(units.rat == "Rat3") & (units.day == "FD4") & (units.type == "ES")].copy()
u3["shuffle_spikes_dropped"] = 1 - u3.null_spikes_used / u3.spikes_in_epoch
wide = u3.pivot_table(index=["tetrode", "cluster"], columns="epoch", values=["spikes_in_epoch", "spikes_used", "info", "null95", "p"], aggfunc="first")
wide.columns = [f"{v}_{'e1' if e == ep_names[0] else 'e2'}" for v, e in wide.columns]
wide = wide[[f"{v}_{k}" for v in ["spikes_in_epoch", "spikes_used", "info", "null95", "p"] for k in ["e1", "e2"]]]
wide["r"] = pd.Series({(t, c): r for (t, c), r in zip(both, same)})
wide = wide.reset_index()
print(f"Units (e1 = {ep_names[0]}, e2 = {ep_names[1]}) with a spike in either epoch; "
      f"{int(((wide.spikes_in_epoch_e1 == 0) & (wide.spikes_in_epoch_e2 == 0)).sum())} units of the file have none:")
print(wide[(wide.spikes_in_epoch_e1 > 0) | (wide.spikes_in_epoch_e2 > 0)].round(3).to_string(index=False))
print("\nPer epoch:")
rows = []
for k, (a, b) in enumerate(zip(es_epochs.start, es_epochs.stop)):
    d = u3[u3.epoch == ep_names[k]]
    t = d.dropna(subset=["info"])
    rows.append({"epoch": ep_names[k], "duration_s": b - a, "occupancy_counts": int(occs[k].sum()),
                 "bins_occ>=5": int((occs[k] >= nl.MIN_OCC).sum()), "units_with_spikes": int((d.spikes_in_epoch > 0).sum()),
                 "tested": len(t), "significant": int((t.p < 0.05).sum()), "median_spikes_used": float(t.spikes_used.median()),
                 "median_info": round(float(t["info"].median()), 3), "median_null_mean": round(float(t.null_mean.median()), 3),
                 "median_null95": round(float(t.null95.median()), 3),
                 "median_shuffle_spikes_dropped": round(float(t.shuffle_spikes_dropped.median()), 3)})
print(pd.DataFrame(rows).to_string(index=False))
in_both = wide.dropna(subset=["spikes_in_epoch_e1", "spikes_in_epoch_e2"])
in_both = in_both[(in_both.spikes_in_epoch_e1 > 0) | (in_both.spikes_in_epoch_e2 > 0)]
d1 = np.diff(es_epochs[["start", "stop"]].values[0])[0]
d2 = np.diff(es_epochs[["start", "stop"]].values[1])[0]
r1, r2 = in_both.spikes_in_epoch_e1.values / d1, in_both.spikes_in_epoch_e2.values / d2
rho = stats.spearmanr(r1, r2).statistic
null_rho = np.array([stats.spearmanr(r1, rng_c4.permutation(r2)).statistic for _ in range(1000)])
print(f"\nUnits with spikes in either epoch: {len(in_both)}; with spikes in both: {int(((r1 > 0) & (r2 > 0)).sum())}; "
      f"in the first only: {int(((r1 > 0) & (r2 == 0)).sum())}; in the second only: {int(((r1 == 0) & (r2 > 0)).sum())}")
print(f"Mean rate, epoch 1 against epoch 2, Spearman rho = {rho:.3f}; re-paired 95th percentile {np.percentile(null_rho, 95):.3f}; "
      f"p = {(np.sum(null_rho >= rho) + 1) / 1001:.4f}")


**Result: reproduced.** In Rat3's two flight day 4 ES epochs, the 15 units with 50 or more spikes in
each have a median rate-map correlation of 0.537 (range -0.107 to 0.849), above the 95th percentile of
re-paired medians (0.102, p = 0.001) and above 0.5; 8 of 15 exceed 0.5. These are the same as in
0.260728.1352. With other smoothing the median is 0.315 to 0.707, always above its re-paired 95th
percentile (p = 0.001), but at 0.5 bins of smoothing it is below 0.5 (0.315 and 0.341), which would
make the claim partially reproduced. The dandiset's own ES1 and ES2 maps, now linked to the same
unit, give a median of 0.340 over 18 units and over the 15 units of the main test. A unit is the same
sorted cluster in both epochs, as the units' description now states (each tetrode was cut once);
whether a cluster is one neuron throughout is not shown, and the files hold no waveforms. The units'
mean rates are related across the two epochs (Spearman rho 0.598 over the 24 units with a spike in
either, re-paired 95th percentile 0.339). Of the file's 47 units, 23 have no spike in either ES
epoch, 18 have spikes in both, 1 only in the first and 5 only in the second. In the first epoch 11
of 16 units are significant against shuffles and in the second 1 of 19. The second epoch has similar
occupancy (62327 against 71496 counts), more spikes per tested unit (median 455 against 301) and
similar shuffle drops (0.215 against 0.205), but its units' information is at the level of their
shuffles (median 1.629 against a shuffle mean of 1.646; in the first epoch 2.155 against 1.667).
Several units whose fields correlate across the epochs are significant in the first epoch only,
and for them the 95th percentile of the shuffles rises more in the second epoch than their information does: TT4
cluster 1 (r = 0.820) has 2.449 against 1.326 in the first epoch and 3.592 against 4.305 in the
second, TT5 cluster 1 (r = 0.838) 0.910 against 0.504 and 1.472 against 2.393, TT8 cluster 5 (r =
0.436) 1.101 against 0.785 and 1.293 against 2.183. The epochs use 348 and 299 bins with 5 or more
occupancy counts. With Poisson trains in place of the spikes the test gives 0 to 2 false positives
of 16 to 20 units in either epoch (flight-spatial-code), so it is not too lenient there. The 95th
percentile of a unit's shuffles follows how few 10 s periods hold its spikes and how few spikes it
has (flight-spatial-code): TT4 cluster 1 has 1582 spikes filling an effective 0.267 of the 10 s bins
in the first epoch and 423 filling 0.096 in the second, and TT5 cluster 1 8844 spikes at 0.575 and
2221 at 0.307. In the second epoch these units fire fewer spikes in fewer periods, so the shuffles
reach higher information, and the test loses power for them; the stability measure does not depend
on the shuffle test. Which sessions Figure 3 compares is not known.

## theta-normal and ripples-normal

> Hippocampal EEG showed normal theta rhythm during locomotion and normal sharp waves and ripples
> during quiet rest in flight. (Dandiset description)

Blocked. No file holds EEG or any continuous voltage: the unfiltered listing in the data check
shows an empty `acquisition/` group and no ElectricalSeries or LFP in any file, and the draft's
experiment description says no EEG or other continuous voltage was archived. The NASA record of the
study (OSD-968) describes its files as spike files, rate-map files, legacy metadata files and NWB
copies, and its metadata does not mention EEG, theta, ripples or LFP (printed under
three-orthogonal-surfaces). Its archives were not opened, so an EEG file inside one cannot be ruled
out.

In [ ]:
print(f"Run time of the notebook: {(time.time() - started) / 60:.1f} minutes")


## Summary

| Claim | Outcome |
|---|---|
| `flight-spatial-code` | Not reproduced: Rat1 FD9 0.40 of units significant (0.40 to 0.44 over the repeats), Rat2 FD9 0.071 |
| `three-orthogonal-surfaces` | Blocked: metadata insufficient (no mapping of pixels to the track's surfaces) |
| `si-comparable-to-preflight` | Reproduced: FD9 1.385 against preflight 1.448 bits/spike, p = 0.45 |
| `fd4-abnormal-selectivity` | Reproduced: FD4 1.177 against preflight 1.441 bits/spike, one-sided p = 0.024 (0.055 with the three maps the draft leaves out) |
| `fig3-stability-across-sessions` | Reproduced: median r 0.537 between Rat3's two FD4 ES epochs (null 95th percentile 0.102) |
| `theta-normal` | Blocked: data missing (no EEG) |
| `ripples-normal` | Blocked: data missing (no EEG) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

Against the first attempt on 0.260728.1352, `fd4-abnormal-selectivity` changes from not reproduced
to reproduced, because the draft leaves out three maps that moved the test across 0.05; the other
outcomes are the same.

How the analysis differs from the paper's. The paper's full text was not available, so its
Methods, tests and thresholds are unknown; every method here is a common choice fixed before the
analysis. Four claims come from the dandiset's description of the paper's findings, and one from a
figure title alone. Spatial information is computed from the spikes for the spike-based claims, with
the occupancy maps of the original analysis, as the dandiset holds position only at spike times;
the comparisons with preflight use the original rate maps, as the preflight files have no position.
The rate maps are smoothed with an unrecorded kernel and their spatial information is not corrected
for the number of spikes; a count-adjusted variant is shown. The stability claim takes a unit to be
the same neuron in two epochs because it is the same sorted cluster. With Poisson trains moved to
position sample times, which reproduce the real drop rates, the shuffle test gives 0.096 false
positives over all flight ES epochs and up to 0.214 in one session (0.074 and 0.160 with plain
Poisson trains), against the 0.05 the criterion assumes; judged against these fractions, the outcome
of flight-spatial-code is the same. Real trains reversed in time, which keep their bursts and changes
in rate, give 2 of 122: with real temporal structure the test is conservative, and it has less power
for units whose spikes fall in a few periods, such as most of Rat2's.

What was found about the dandiset (draft).
- Each flight file now holds the shared tracker's trace only in its own rat's task epochs, and says
  the trace there is taken to be that rat's. Rat2's units are not selective with it in any window,
  and Rat1's are selective in Rat2's FD4 ES window, so whose position the trace is in Rat2's epochs
  is still not established.
- The dandiset description still says the preflight recordings were on a flat rectangular track,
  while the preflight epochs and rate maps are labelled ES and MC, and still reports EEG, which no
  file holds.
- Rat2's preflight file shares no tetrode with its flight files, and the assignment of the preflight
  recordings to rats is an open question in the conversion notes.
- Position is given only at spike times, so occupancy cannot be computed from the files' position.
- The maps are smoothed with a kernel that is not recorded, and the rats' age, the camera view and
  the pixel size are not recorded.
- Fixed since the first attempt: the rate maps are linked to their units; NaN replaces -999 and the
  lost position, and the [63, 0] occupancy is documented; identical maps are left out (all copies,
  so six maps are gone); units list their spike files and windows and are documented as one sorted
  cluster across epochs; the spike-time resolution is documented and explains the zero intervals;
  the epochs no longer overlap.